In [ ]:
# Setup: in Google Colab, download and install the repository (a few minutes, once per session); elsewhere, work from the
# repository. Pages that read the derived catalogues need Zenodo bundle 1 (python scripts/fetch_release.py).
import os, subprocess, sys
os.environ['MPLBACKEND'] = 'Agg'   # as in a plain script run: Jupyter's inline settings would change the saved figures
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/lmc-ancient-disc'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vasilybelokurov/lmc-ancient-disc', '/content/lmc-ancient-disc'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/lmc-ancient-disc'], check=True)
    os.chdir('/content/lmc-ancient-disc')
else:
    while not os.path.exists('pyproject.toml'):
        os.chdir('..')
sys.path[:0] = ['src', 'scripts', 'kb']
if not os.path.exists('data/giants_apogee.parquet'):   # the catalogues of Zenodo bundle 1, checked against data/sources.yaml
    subprocess.run([sys.executable, 'scripts/fetch_release.py'], check=True)

# Kinematics: velocity field and disc model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/lmc-ancient-disc/blob/main/kb/notebooks/chapters/kinematics_model.ipynb)

Every number on this page is written by `scripts/paper_text_kin_model.py` from the result tables and the posterior samples of the
kinematic fits (`results/macros_kinmodel.tex`); the code cells are the figure scripts themselves (`scripts/paper_kin_residuals.py`,
`scripts/paper_kin_ellipsoid_mock.py`, `scripts/paper_kin_corner.py`), step by step. The fits themselves
(`scripts/fit_kinematics_mcmc.py`) take hours and are not rerun here; their results are read from `results/`. The fit script and the
scripts of the checks and mocks are shown step by step where they enter the story, without being run.

The Large Magellanic Cloud (LMC) is not a disc in equilibrium. Its outer disc is lopsided and truncated at different radii in
different directions, its bar is offset from the centre of the outer isophotes (the contours of constant surface brightness), and
the Small Magellanic Cloud (SMC) and the Milky Way perturb it today ([van der Marel 2001](https://doi.org/10.1086/323100);
[Choi et al. 2018](https://doi.org/10.3847/1538-4357/aae083); [Belokurov & Erkal 2019](https://doi.org/10.1093/mnrasl/sly178);
[Cullinane et al. 2022a](https://doi.org/10.1093/mnras/stab3350); [Cullinane et al. 2022b](https://doi.org/10.1093/mnras/stac733)).
A rigid disc model will not describe every star, and we do not try to build one. We ask how strongly each metallicity group of the
old population is supported by rotation, and we answer with the simplest disc model that describes all groups in the same way: a
thin disc whose stars have a mean rotation and a velocity ellipsoid (the three-dimensional Gaussian distribution of the stars'
random velocities about the mean) aligned with the disc. This page shows the measured velocity field, then the model and what it
leaves in the data, and, in the checks at the end, the tests of the model on mocks and the simpler model of our first fits. The
rotational support of each group, the centre and geometry, and the sensitivity to the metallicity errors and to the model are the
subject of chapter 7.

## The measured velocity field

The old population is seen through two tracers. The red giants have proper motions (angular motions on the sky, in mas/yr) from
[Gaia DR3](https://doi.org/10.1051/0004-6361/202243940) and line-of-sight velocities from the spectroscopic surveys
[APOGEE](https://doi.org/10.3847/1538-3881/aa784d) (Data Release 17; [Abdurro'uf et al. 2022](https://doi.org/10.3847/1538-4365/ac4414))
or SDSS-V BOSS ([Kollmeier et al. 2026](https://doi.org/10.3847/1538-3881/ae0576); Data Release 20,
[SDSS Collaboration 2026](https://doi.org/10.48550/arXiv.2607.26149)); a star observed by both surveys keeps its APOGEE values. The
samples are built in the data chapters. Each giant is placed in one of five bins of its measured iron abundance [Fe/H] (the
logarithm of the iron-to-hydrogen ratio relative to the Sun's): below -1.5, -1.5 to -1.1,
-1.1 to -0.8, -0.8 to -0.5 and above -0.5. The RR Lyrae
(old pulsating stars of type ab, LMC members from the OGLE-IV collection, [Soszyński et al. 2016](https://doi.org/10.48550/arXiv.1606.02727),
and the Gaia DR3 catalogue, [Clementini et al. 2023](https://doi.org/10.1051/0004-6361/202243964)) have Gaia proper motions only.
We use the stars at angular distances 2° < ρ < 9° from the LMC centre of
[van der Marel & Kallivayalil (2014)](https://doi.org/10.1088/0004-637X/781/2/121) (hereafter vdMK14; right ascension and declination
(α, δ) = (79.88°, -69.59°)): the inner limit excludes the bar, the outer one the sparse periphery. These
samples hold 2 168, 3 542, 10 155, 28 909 and 11 657 giants, from
metal-poor to metal-rich, and 14 707 RR Lyrae; the RR Lyrae kinematics we quote are measured at
4° < ρ < 9° (7 200 stars; chapter 7).

From each velocity we subtract the bulk motion of the LMC as seen at the star's position, which removes the systemic motion and its
perspective effects, and we convert the proper motions to velocities (km/s) at the distance where the star's line of sight meets the
disc plane. We split the velocity across the sky into a tangential component $v_t$, pointing 90° east of the direction away from the
centre, and a radial component $v_r$, pointing away from the centre. These velocities depend on the frame: the centre about which
$v_t$ and $v_r$ are taken, the systemic motion subtracted and the disc plane onto which the stars are placed. We draw all maps and
profiles in one frame, that of the joint fit of all groups (stage 2 of the model below, Table 6.2), so that a difference
between two columns of Figure 6.1–Figure 6.3 is a difference between the stars and
not between the frames; had each group been drawn in the frame of its own fit, the centre of a giant bin would move by up to
0.3° and its inclination by up to 3.3°. The numbers we quote for each group come instead from its own
fit (stage 1); the frame of vdMK14 is compared in the section on what the models leave in the data.

The top rows of Figure 6.1, Figure 6.2 and Figure 6.3 map
$v_t$, $v_r$ and the line-of-sight velocity $v_{\rm los}$ of each giant bin and of the RR Lyrae, and Figure 6.4
plots the same velocities against position angle (the angle on the sky about the centre, measured from north through east). All
groups rotate in the same sense, clockwise on the sky, so $v_t<0$ (at 3° < ρ < 7° it ranges over
position angle from −41 to −83 km/s in the harmonic fits described below; a negative residual in $v_t$ therefore means
faster rotation than the model), and the line-of-sight velocity is largest at the two ends of the line of nodes (the line where the
disc plane crosses the plane of the sky), with opposite signs, and changes sign across the minor axis (perpendicular to the line of
nodes). The metal-rich giants rotate faster than the metal-poor giants and the RR Lyrae at almost every position angle
(Figure 6.4a). We describe the variation of a velocity $v$ with position angle $\psi$ by a harmonic fit
to the stars, $v=c_0+\sum_m A_m\cos[m(\psi-\psi_m)]$, where $A_m$ is the amplitude of the $m$-th term and $\psi_m$ the position angle
of its maximum: $m=1$ is a lopsided (one-sided) pattern, $m=2$ a two-fold one. Both $v_t$ and $v_r$ carry an $m=2$ term of
10–14 km/s (harmonic fits to the stars at 3° < ρ <
7°) with the same phase in every group, as the projection of an inclined rotating disc does; about the fitted
centre the $m=1$ term of $v_t$ is only 2–7 km/s.

The script below computes the residuals of every star about each model, the maps, the profiles and the harmonic fits. It is run
with the argument `ellipsoid` (the velocity-ellipsoid model of the next section); without it, it draws the same figures for the
sky-aligned model of the checks.

In [ ]:
import sys
sys.argv = ['paper_kin_residuals.py', 'ellipsoid']   # as: python scripts/paper_kin_residuals.py ellipsoid

<details><summary>About <code>scripts/paper_kin_residuals.py</code></summary>

```text
Paper figures: the measured velocity field of the old population and what the disc models leave in it (kinematics section).

Every star's proper motion and line-of-sight velocity is compared with a model velocity from lmcold.kinfit.model; the difference
is projected on the sky-polar basis about the model's centre (tangential v_t, 90 deg east of the direction away from the centre,
and radial v_r; lmcold.kinfit.residual_polar, each star at its disc distance) and, for the giants, along the line of sight (v_los).
Models, as parameter vectors of kinfit.PARAMS:
- 'data': the frame of the joint Level-2 fit without internal motion (its centre, systemic proper motion and velocity, inclination
  and line of nodes; v0 = 0, which also removes the streaming, proportional to v0). The difference is the internal velocity field
  with the bulk motion and its perspective removed, in the polar basis about the fitted centre: the data.
- 'model': the internal velocity field of Level 2 in the same frame ('data' minus 'L2' residuals, i.e. L2 model minus frame).
- 'pull': the 'L2' residual of each star divided by the scatter Level 2 predicts for it in that component: sqrt(sigma^2 +
  k^2 e^2) for v_t and v_r (sigma the population's proper-motion dispersion, k its Gaia error scale, e the star's proper-motion
  error ellipse projected on the component, in km/s at its disc distance) and sqrt(sigma_los^2 + evlos^2) for v_los, as in the
  likelihood (lmcold.kinfit.loglike_stars). N(0, 1) per star if Level 2 is right.
- 'vdMK14': the vdMK14 geometry and systemic motion (lmcold.constants.LMC) with each population's rotation curve, dispersions,
  error scale and outlier fraction fitted in that frame (maximum likelihood, kinfit.fit with VDMK14_FREE).
- 'L1': the joint symmetric-disc fit (results/kin_mcmc_data_centre.csv, stage 2: shared centre, systemic motion, i, theta; each
  population's own v0, R0); 'L2': the joint fit with streaming (results/kin_mcmc_data_stream.csv, stage 2, shared m = 2 terms).
  Posterior medians.
Stars: RR Lyrae members (proper motions only; lmcold.samples.rrab) and the giants in their five measured [Fe/H] bins
(lmcold.samples.giants), at 2 < rho < 9 deg from the vdMK14 centre (the fitted range); a star whose L2 residual exceeds CLIP km/s
(plus three times its own error for the proper-motion components) is dropped from every panel (Milky Way stars and gross errors;
2-6 per cent in the metal-poor bins, < 1 per cent above -1.1).

Writes (all panels equal; maps: x east to the left, y north [deg] on the orthographic plane about the joint Level-2 centre,
lmcold.geometry.tangent_plane_coords):
- figures/paper_kin_maps_vt.png, paper_kin_maps_vr.png, paper_kin_maps_vlos.png (Sects. 4.1, 4.3): one figure per component;
  rows 'data', 'model' and the significance of the residual: z = median pull x sqrt(n) / 1.2533 per cell of n stars (the
  median of n unit normals has standard deviation 1.2533 / sqrt(n), so z is N(0, 1) per cell if Level 2 is right; the median
  keeps the remaining Milky Way stars from dominating a cell); columns the giant bins metal-poor to metal-rich and the RR Lyrae (not for
  v_los: no velocities); median per CELL-deg cell with >= NMIN stars; colour VLIM for data and model, -ZLIM..ZLIM for z;
  vdMK14 centre (+, the origin of rho for the star selection), Level-2 line of nodes (dashed).
- figures/paper_kin_data_profiles.png (Sect. 4.1): 3 x 1 panels, v_t, v_r, v_los of 'data' against position angle (north through
  east about the Level-2 centre) at 3 < rho < 7 deg: median per 30-deg bin, one line per population; line of nodes (dotted) and
  minor axis (dash-dotted).
- figures/paper_kin_residual_profiles.png (Sect. 4.3): 3 x 3 panels; rows v_t, v_r, v_los residuals against position angle (about
  each model's centre) at 3 < rho < 7 deg; columns 'vdMK14', 'L1', 'L2'; one line per population; in each panel the range over
  the giant bins of the m = 1 and m = 2 amplitudes of a harmonic fit (lmcold.harmonics) to the stars.
- results/kin_residual_profiles.csv: population, model, component, n, mean, A1, phase1, A2, phase2 [km/s, deg] (harmonic fit at
  3 < rho < 7 deg; phase = position angle of the maximum).
- results/kin_residual_cells.csv: population, component, n_cells, z_sd (standard deviation of the cell significances z of the
  maps), z_gt3 (fraction of cells with |z| > 3).
- results/kin_residual_radial.csv and figures/paper_kin_residual_radial.png (appendix): median Level-2 residual of each component
  [km/s] in rings RADIAL_EDGES [deg] about the Level-2 centre, error 1.2533 x standard deviation / sqrt(n); one line per
  population, three equal panels (v_t, v_r, v_los).

With the argument 'ellipsoid', 'L2' is the joint fit of the disc-frame velocity-ellipsoid model (results/kin_mcmc_data_ellipsoid.csv,
stage 2: Level-2 mean field with the outer slope vslope, dispersions sigma_R, sigma_phi, sigma_z with radial gradient), the pull
uses the scatter that ellipsoid predicts for each star (its intrinsic covariance, lmcold.kinfit.intrinsic_cov, projected on t, r and
the line of sight), and every output name above gets the suffix '_ellipsoid'. The RR Lyrae own parameters (RRL_OWN) are refitted
(maximum likelihood) on their stars at OUTER_INNER < rho < 9 deg with the joint geometry and streaming held: over 2-9 deg one Gaia
error scale cannot describe the crowded inner disc and the fit drives their dispersion gradient to -0.57 per kpc, so their quoted
values come from 4-9 deg (scripts/paper_kin_support_ellipsoid.py). All RR Lyrae at 2-9 deg are then shown with these parameters.

Inputs: the giant and RR Lyrae catalogues of lmcold.samples (Zenodo bundle 1) and the joint-fit tables results/kin_mcmc_data_centre.csv,
kin_mcmc_data_stream.csv and kin_mcmc_data_ellipsoid.csv (scripts/fit_kinematics_mcmc.py). The paper and KB chapter 6 use both runs:
with 'ellipsoid' (main text) and without (the sky-aligned model of the checks).

Run: .venv/bin/python scripts/paper_kin_residuals.py [ellipsoid]
```
</details>

**Data read by `scripts/paper_kin_residuals.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/giants_apogee.parquet` | Zenodo bundle 1 | APOGEE DR17 giants with Gaia DR3 |
| `data/giants_boss.parquet` | Zenodo bundle 1 | SDSS-V DR20 BOSS giants with Gaia DR3, on the APOGEE scale |
| `data/rrab.parquet` | Zenodo bundle 1 | RRab within 20 deg: OGLE-IV + Gaia DR3, [Fe/H], dmu, membership |
| `results/kin_mcmc_data_centre.csv` | repository | posterior summary of the free-centre model; MCMC; written by `python scripts/fit_kinematics_mcmc.py data centre` |
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |
| `results/kin_mcmc_data_stream.csv` | repository | posterior summary of the sky-aligned dispersion model with m = 2 streaming, 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data stream` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binned_statistic_2d

from fit_kinematics_mcmc import OUTER_INNER, ring
from lmcold import geometry, harmonics, kinfit, samples
from lmcold.constants import KAPPA, LMC
from lmcold.footprint import angular_distance
from lmcold.layout import panel_grid
from lmcold.sources import ROOT

### Settings and helper functions
The catalogue columns, the clipping of outliers (`CLIP`), the map cells (`CELL`, `NMIN`), the ring of the position-angle profiles
(`PROFILE_RING`), the colour ranges, and the model whose residuals are drawn: with the argument `ellipsoid` the joint fit of the
velocity-ellipsoid model (`results/kin_mcmc_data_ellipsoid.csv`), otherwise that of the sky-aligned model. `params()` reads the
stage-2 posterior medians of a group, `residuals()` gives each star's residual velocities about a model, and `predicted_sd()` the
scatter the model predicts for each of them.

In [ ]:
COLS = ('ra', 'dec', 'pmra', 'pmdec', 'pmra_error', 'pmdec_error', 'pmra_pmdec_corr', 'vlos', 'evlos')   # kinfit.data() inputs
CLIP = 150.0  # km/s: residual beyond which a star is dropped (plus three times its own error for proper motions)
CELL, NMIN = 1.0, 5  # deg, minimum stars per map cell
PROFILE_RING = (3.0, 7.0)  # deg, stars used in the position-angle profiles
PA_EDGES = np.arange(0, 361, 30)  # deg: position-angle bins of the profiles
COMPS = ('v_t', 'v_r', 'v_los')   # residual components: tangential, radial, line of sight
COLOURS = ['#d7191c', '#fdae61', '#a6d96a', '#1a9641', '#2878b5', '#7b3294']  # giant bins metal-poor first, RR Lyrae (corner plots)
# colour range [km/s] and map of the data and model rows: v_t is negative everywhere (clockwise rotation), so a sequential map
VLIM = dict(v_t=(-100, 0, 'viridis'), v_r=(-50, 50, 'RdBu_r'), v_los=(-50, 50, 'RdBu_r'))
ZLIM = 4.0  # colour range of the residual significance z
LABELS = {'v_t': r'$v_t$', 'v_r': r'$v_r$', 'v_los': r'$v_{\rm los}$'}   # axis labels of the components
VDMK14_FREE = ('v0', 'R0', 'sigma', 'k', 'logit_fout')  # fitted in the vdMK14 frame (plus sigma_los for the giants)
RRL_OWN = ('v0', 'R0', 'vslope', 'k', 'logit_fout') + kinfit.ELLIPSOID  # RR Lyrae, refitted at 4-9 deg in the joint frame (ellipsoid)
RADIAL_EDGES = np.array([2., 3., 4., 5., 6., 7., 7.5, 9.5])  # deg from the Level-2 centre, radial residual profile
ELLIPSOID = 'ellipsoid' in sys.argv[1:]          # command-line argument: use the velocity-ellipsoid run as Level 2
SUFFIX = '_ellipsoid' if ELLIPSOID else ''       # appended to every output name
L2_RUN = 'ellipsoid' if ELLIPSOID else 'stream'  # results/kin_mcmc_data_<L2_RUN>.csv
# Column titles of the residual-profile figure.
MODEL_TITLES = dict(vdMK14='geometry of vdMK14', L1='Level 1 (symmetric disc)',
                    L2='Level 2, velocity ellipsoid' if ELLIPSOID else 'Level 2 (+ streaming)')


def params(csv, pop):
    """Stage-2 posterior medians of results/<csv>: shared parameters plus population pop's own, as a kinfit.PARAMS vector."""
    t = pd.read_csv(ROOT / 'results' / csv)   # columns stage, population, param, median, p16, p84
    t = t[(t.stage == 2) & t.population.isin(['shared', pop])]
    # Start from kinfit.start() (values of parameters not in the fit) and insert each fitted median.
    p = kinfit.start()
    for _, r in t.iterrows():
        p[kinfit.PARAMS.index(r.param)] = r['median']
    return p


def residuals(p, c):
    """Residuals (observed - model) of catalogue dict c for parameter vector p: position angle about p's centre [deg] and
    v_t, v_r, v_los [km/s] (v_los NaN without a velocity)."""
    pa, vt, vr = kinfit.residual_polar(p, c)   # PA [deg], v_t, v_r residuals [km/s], shape (N,) each
    # Line-of-sight residual: observed minus model v_los (third output of kinfit.model) [km/s].
    d = kinfit.data(*(c[k] for k in COLS))
    vl = np.asarray(c['vlos']) - np.asarray(kinfit.model(p, d['rhat'], d['ea'], d['ed'])[2])
    return pa, dict(v_t=vt, v_r=vr, v_los=vl)


def predicted_sd(p, c):
    """Scatter [km/s] that parameter vector p predicts for each star's v_t, v_r and v_los residuals: intrinsic dispersion plus
    the k-scaled measurement errors, as in kinfit.loglike_stars. residual_polar is linear in the proper motion, so the change of
    its output for a unit change of pmra (pmdec) gives each star's km/s per mas/yr along t and r (kappa D times the east (north)
    component of the unit vector)."""
    # Residuals as observed and with pmra or pmdec raised by 1 mas/yr: their differences are the Jacobian (km/s per mas/yr).
    _, vt, vr = kinfit.residual_polar(p, c)
    _, vt_a, vr_a = kinfit.residual_polar(p, c | dict(pmra=c['pmra'] + 1.0))
    _, vt_d, vr_d = kinfit.residual_polar(p, c | dict(pmdec=c['pmdec'] + 1.0))
    # Proper-motion errors [mas/yr] and their covariance [(mas/yr)^2].
    sa, sd = c['pmra_error'], c['pmdec_error']
    cov = c['pmra_pmdec_corr'] * sa * sd
    # Sky-aligned model: PM dispersion sigma [km/s], Gaia error scale k, line-of-sight dispersion sigma_los [km/s].
    sigma, k, sigma_los = (p[kinfit.PARAMS.index(n)] for n in ('sigma', 'k', 'sigma_los'))
    if ELLIPSOID:
        # Intrinsic variance along a sky direction: (je, jn) / (kappa D) is its unit vector in (east, north).
        d = kinfit.data(*(c[k] for k in COLS))
        C = np.asarray(kinfit.intrinsic_cov(p, d))
        kD = KAPPA * np.asarray(kinfit.model(p, d['rhat'], d['ea'], d['ed'])[3])
        intrinsic = lambda je, jn: (je ** 2 * C[:, 0, 0] + jn ** 2 * C[:, 1, 1] + 2 * je * jn * C[:, 0, 1]) / kD ** 2  # noqa: E731
        intrinsic_los = C[:, 2, 2]
    else:
        # Sky-aligned model: the same isotropic variance sigma^2 in every sky direction.
        intrinsic = lambda je, jn: sigma ** 2  # noqa: E731
        intrinsic_los = sigma_los ** 2
    # Predicted SD [km/s] of v_t and v_r: intrinsic variance + k^2 x the PM error ellipse projected with the Jacobian (je, jn).
    out = {}
    for comp, (je, jn) in (('v_t', (vt_a - vt, vt_d - vt)), ('v_r', (vr_a - vr, vr_d - vr))):
        out[comp] = np.sqrt(intrinsic(je, jn) + k ** 2 * (je ** 2 * sa ** 2 + jn ** 2 * sd ** 2 + 2 * je * jn * cov))
    out['v_los'] = np.sqrt(intrinsic_los + np.asarray(c['evlos']) ** 2)   # v_los: intrinsic plus velocity error [km/s]
    return out

### Load the data
The giant catalogues of Zenodo bundle 1 through `lmcold.samples.giants()`, merged and split into the five [Fe/H] bins, and the
RR Lyrae members through `lmcold.samples.rrab()` (from `data/rrab.parquet`; proper motions only, so their line-of-sight velocity
columns are set to NaN), each cut to $2<\rho<9^\circ$ by `ring`. The joint fits are read inside the loop below by `params()`:
the stage-2 posterior medians of `results/kin_mcmc_data_centre.csv` (Level 1) and of `results/kin_mcmc_data_stream.csv` (Level 2;
with the argument `ellipsoid`, `results/kin_mcmc_data_ellipsoid.csv`).

In [ ]:
# Giants: (name, catalogue dict) for each [Fe/H] bin, from the release catalogues data/giants_apogee.parquet, giants_boss.parquet.
cats = [(f'giants {lab}', ring(c)) for lab, c in samples.giants()]
# RR Lyrae members from data/rrab.parquet, without line-of-sight velocities.
cats.append(('RR Lyrae', ring(samples.rrab() | dict(vlos=np.full(len(samples.rrab()['ra']), np.nan),
                                                     evlos=np.full(len(samples.rrab()['ra']), np.nan)))))

### Residuals of every star about each model
For each giant [Fe/H] bin and the RR Lyrae at 2-9 deg: the residuals about the joint frame without rotation (the data), about
the vdMK14 frame (rotation curve and dispersions refitted by maximum likelihood), about the joint Level-1 and Level-2 fits, and the
residual divided by its predicted scatter (the pull). With `ellipsoid`, the RR Lyrae parameters are first refitted at 4-9 deg in
the joint frame. Stars with residuals beyond `CLIP` are dropped, and each component is fitted with an $m=1$ plus $m=2$ harmonic
series in position angle (`lmcold.harmonics`).

In [ ]:
vdmk14 = kinfit.start()  # vdMK14 centre, systemic motion, i, theta (lmcold.constants.LMC)
res, rows = {}, []   # per-population results for the figures; rows of the harmonic table
for name, c in cats:
    # Level-1 and Level-2 parameter vectors of this population (stage-2 posterior medians).
    l1, l2 = params('kin_mcmc_data_centre.csv', name), params(f'kin_mcmc_data_{L2_RUN}.csv', name)
    # Ellipsoid run: RR Lyrae own parameters refitted on stars at OUTER_INNER (4) to 9 deg, joint frame held (docstring).
    if ELLIPSOID and name == 'RR Lyrae':
        o = ring(c, OUTER_INNER)
        l2 = kinfit.fit(kinfit.data(*(o[k] for k in COLS)), free=RRL_OWN, p0=l2, ellipsoid=True)[0]
        print('RR Lyrae refitted at 4-9 deg:', {n: round(float(l2[kinfit.PARAMS.index(n)]), 3) for n in RRL_OWN}, flush=True)
    # 'data' frame: Level 2 with v0 = 0 (no rotation, no streaming).
    data_p = l2.copy()
    data_p[kinfit.PARAMS.index('v0')] = 0.0
    # vdMK14 frame: rotation curve, dispersions, error scale, outlier fraction (and sigma_los if velocities) fitted by ML.
    los = ('sigma_los',) if np.isfinite(c['vlos']).any() else ()
    vdmk14_p = kinfit.fit(kinfit.data(*(c[k] for k in COLS)), free=VDMK14_FREE + los, p0=vdmk14)[0]
    # Residuals (PA [deg], {component: km/s}) about each model; 'model' = data minus L2 residual = L2 model minus frame.
    out = {m: residuals(p, c) for m, p in (('data', data_p), ('vdMK14', vdmk14_p), ('L1', l1), ('L2', l2))}
    out['model'] = (out['data'][0], {k: out['data'][1][k] - out['L2'][1][k] for k in COMPS})
    # Pull: Level-2 residual / its predicted SD (dimensionless).
    sd = predicted_sd(l2, c)
    out['pull'] = (out['L2'][0], {k: out['L2'][1][k] / sd[k] for k in COMPS})
    # Proper-motion components: CLIP plus three times the star's own error [km/s] (faint RR Lyrae errors reach 100 km/s).
    # D: disc distance of each star [kpc] (fourth output of kinfit.model); epm: PM error in km/s at that distance.
    D = np.asarray(kinfit.model(l2, *(kinfit.data(*(c[k] for k in COLS))[k] for k in ('rhat', 'ea', 'ed')))[3])
    epm = KAPPA * D * np.hypot(c['pmra_error'], c['pmdec_error'])
    vv = out['L2'][1]   # Level-2 residuals [km/s]
    # Clipped stars (the same mask is applied to every model and panel).
    bad = (np.abs(vv['v_t']) > CLIP + 3 * epm) | (np.abs(vv['v_r']) > CLIP + 3 * epm) | (np.abs(np.nan_to_num(vv['v_los'])) > CLIP)
    rho = angular_distance(c['ra'], c['dec'], LMC['ra0'], LMC['dec0'])   # [deg] from the vdMK14 centre
    # Stars of the position-angle profiles: PROFILE_RING (3-7 deg), not clipped.
    prof = (rho > PROFILE_RING[0]) & (rho < PROFILE_RING[1]) & ~bad
    cra, cdec = (float(v) for v in kinfit.centre(*l2[:2]))  # joint Level-2 centre [rad], the same for every population
    # Orthographic tangent-plane coordinates about the Level-2 centre [deg]: x east, y north.
    x, y = (np.rad2deg(np.asarray(v)) for v in geometry.tangent_plane_coords(np.deg2rad(c['ra']), np.deg2rad(c['dec']), cra, cdec))
    # Kept for plotting: positions, profile mask, residuals of every model (clipped stars removed), vdMK14 centre position [deg]
    # on the same plane, and the Level-2 line of nodes theta [rad].
    res[name] = dict(x=x[~bad], y=y[~bad], prof=prof[~bad], out={m: (pa[~bad], {k: v[~bad] for k, v in vv.items()})
                                                                  for m, (pa, vv) in out.items()},
                     vdmk14=[float(np.rad2deg(v)) for v in geometry.tangent_plane_coords(np.deg2rad(LMC['ra0']), np.deg2rad(LMC['dec0']),
                                                                                         cra, cdec)],
                     theta=np.deg2rad(l2[kinfit.PARAMS.index('theta')]))
    # Harmonic fit (m <= 2) of each component of the residual models 'data', 'vdMK14', 'L1', 'L2', over >= 50 profile stars.
    for m, (pa, vv) in out.items():
        if m in ('model', 'pull'):
            continue
        for comp, v in vv.items():
            k = prof & ~bad & np.isfinite(v)
            if k.sum() < 50:
                continue
            # Coefficients [c0, a1, b1, a2, b2] [km/s]; amplitude [km/s] and PA of the maximum [deg] of m = 1 and m = 2.
            cc, _ = harmonics.fit(pa[k], v[k])
            (A1, P1), (A2, P2) = harmonics.amplitude_phase(cc, 1), harmonics.amplitude_phase(cc, 2)
            rows.append(dict(population=name, model=m, component=comp, n=int(k.sum()), mean=cc[0], A1=A1, phase1=P1, A2=A2, phase2=P2))
    print(f'{name:28s} N {len(c["ra"]):6d}, dropped {bad.sum():5d} ({100 * bad.mean():.1f}%)', flush=True)

### Harmonic amplitudes and plotting functions
The table of harmonic amplitudes and phases (`results/kin_residual_profiles*.csv`), the line of nodes of each frame, and the two
drawing functions: `maps()` for the data, model and significance rows of one component, `profiles()` for the medians against
position angle.

In [ ]:
# Harmonic amplitudes and phases of every population, model and component.
harm = pd.DataFrame(rows)
harm.to_csv(ROOT / f'results/kin_residual_profiles{SUFFIX}.csv', index=False, float_format='%.3f')
names = [n for n, _ in cats]   # population names, giant bins metal-poor first, then RR Lyrae
# Line of nodes [deg] of each frame (shared by all populations in the joint fits), for the reference lines of the profiles.
THETA = dict(vdMK14=LMC['theta'], L1=l1[kinfit.PARAMS.index('theta')], L2=l2[kinfit.PARAMS.index('theta')],
             data=l2[kinfit.PARAMS.index('theta')])
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
edges = np.arange(-10.0, 10.01, CELL)  # the 9-deg selection about vdMK14 reaches 9.5 deg from the Level-2 centre
centres_pa = 0.5 * (PA_EDGES[1:] + PA_EDGES[:-1])   # PA bin centres [deg]


def maps(comp):
    """Maps of component comp: rows data, Level-2 model, residual; one column per population with this component."""
    pops = [n for n in names if np.isfinite(res[n]['out']['data'][1][comp]).any()]   # RR Lyrae dropped for v_los
    size = (7.0 - 0.55 - 0.75 - 5 * 0.05) / 6  # panel width [inch] of the six-column figures
    fig, ax = panel_grid(3, len(pops), width=0.55 + 0.75 + len(pops) * size + (len(pops) - 1) * 0.05, left=0.55, right=0.75,
                         hgap=0.05, vgap=0.05, top=0.3)
    # Rows: (model key, row label, (vmin, vmax, colour map)).
    rows = (('data', 'data', VLIM[comp]), ('model', 'Level-2 ellipsoid model' if ELLIPSOID else 'Level-2 model', VLIM[comp]),
            ('pull', 'residual significance', (-ZLIM, ZLIM, 'RdBu_r')))
    for j, name in enumerate(pops):
        r = res[name]
        for i, (m, lab, (vmin, vmax, cmap)) in enumerate(rows):
            a, v = ax[i, j], r['out'][m][1][comp]
            ok = np.isfinite(v)
            # Median and number of stars per CELL x CELL deg cell (shape: len(edges) - 1 squared).
            med = binned_statistic_2d(r['x'][ok], r['y'][ok], v[ok], 'median', bins=[edges, edges]).statistic
            n = binned_statistic_2d(r['x'][ok], r['y'][ok], v[ok], 'count', bins=[edges, edges]).statistic
            if m == 'pull':
                med = med * np.sqrt(n) / 1.2533  # significance z of the cell's median pull
                z = med[n >= NMIN]
                CELLS.append(dict(population=name, component=comp, n_cells=len(z), z_sd=np.std(z), z_gt3=np.mean(np.abs(z) > 3)))
            # Cells with fewer than NMIN stars blank; transpose because binned_statistic_2d indexes [x, y].
            im = a.pcolormesh(edges, edges, np.where(n >= NMIN, med, np.nan).T, cmap=cmap, vmin=vmin, vmax=vmax,
                              rasterized=True)
            # Level-2 line of nodes through the origin (PA theta, north through east) and the vdMK14 centre (+).
            th = r['theta']
            a.plot(10 * np.sin(th) * np.array([-1, 1]), 10 * np.cos(th) * np.array([-1, 1]), 'k--', lw=0.5)
            a.plot(*r['vdmk14'], '+', color='k', ms=5, mew=0.7)
            a.set(xlim=(10, -10), ylim=(-10, 10), aspect='equal', xticks=[5, 0, -5], yticks=[-5, 0, 5])
            a.tick_params(labelsize=5)
            # Column titles on the top row; tick labels only on the outer panels.
            if i == 0:
                a.text(0.5, 1.04, name.replace('giants ', ''), transform=a.transAxes, ha='center', fontsize=6)
            if i < 2:
                a.tick_params(labelbottom=False)
            else:
                a.set_xlabel('east [deg]', fontsize=6)
            if j:
                a.tick_params(labelleft=False)
            else:
                a.set_ylabel(f'{lab}\nnorth [deg]', fontsize=6)
            if j == len(pops) - 1 and i != 1:
                # colour bars right of the last column: data and model share one (spanning rows 1-2), the significance its own
                cax = a.inset_axes([1.08, -1.05 if i == 0 else 0.0, 0.07, 2.05 if i == 0 else 1.0])
                fig.colorbar(im, cax=cax)
                cax.tick_params(labelsize=5)
                cax.set_ylabel(LABELS[comp] + r' [km s$^{-1}$]' if i == 0 else f'$z$ ({LABELS[comp]} residual)', fontsize=6)
    fig.savefig(ROOT / f'figures/paper_kin_maps_{comp.replace("_", "")}{SUFFIX}.png')
    plt.close(fig)


def profiles(ax_col, model, letters, harmonics_text):
    """Median v_t, v_r, v_los of model per PA bin at PROFILE_RING, one line per population, in the axes column ax_col."""
    for i, comp in enumerate(COMPS):
        a = ax_col[i]
        for name, colour in zip(names, COLOURS):
            pa, vv = res[name]['out'][model]
            k = res[name]['prof'] & np.isfinite(vv[comp])
            if not k.any():
                continue
            # PA bin of each star; median per bin with >= 10 stars, else NaN.
            b = np.digitize(pa[k], PA_EDGES) - 1
            med = [np.median(vv[comp][k][b == j]) if (b == j).sum() >= 10 else np.nan for j in range(len(centres_pa))]
            a.plot(centres_pa, med, 'o-', ms=2.2, lw=0.8, color=colour, label=name.replace('giants ', 'giants, '))
        th = THETA[model]  # the line of nodes of the frame the residuals are taken in
        # Line of nodes (dotted) and minor axis (dash-dotted) at theta, theta + 180 and theta -+ 90 deg.
        for pa0, ls in ((th, ':'), (th + 180, ':'), (th - 90, '-.'), (th + 90, '-.')):
            a.axvline(pa0 % 360, color='0.6', lw=0.5, ls=ls)
        a.axhline(0, color='0.6', lw=0.5)
        a.set(xlim=(0, 360), xticks=np.arange(0, 361, 60))
        a.text(0.02, 0.92, f'({letters[i]})', transform=a.transAxes, fontsize=6)
        # Range over the giant bins of the m = 1 and m = 2 amplitudes [km/s] of this model and component.
        if harmonics_text:
            h = harm[(harm.model == model) & (harm.component == comp) & harm.population.str.startswith('giants')]
            a.text(0.98, 0.05, f'giants: $A_1$ = {h.A1.min():.0f}$-${h.A1.max():.0f}, $A_2$ = {h.A2.min():.0f}$-${h.A2.max():.0f}'
                   r' km s$^{-1}$', transform=a.transAxes, ha='right', fontsize=5)

### Maps of $v_t$, $v_r$ and $v_{\rm los}$
One figure per component; the statistics of the cell significances go to `results/kin_residual_cells*.csv`.

In [ ]:
CELLS = []   # filled by maps(): statistics of the cell significances z per population and component
# One map figure per component (v_t, v_r, v_los).
for comp in COMPS:
    maps(comp)
# Write and print the cell statistics.
pd.DataFrame(CELLS).to_csv(ROOT / f'results/kin_residual_cells{SUFFIX}.csv', index=False, float_format='%.3f')
print(pd.DataFrame(CELLS).round(2).to_string(index=False))

### Radial profile of the residuals
Median Level-2 residual of each component in rings about the Level-2 centre (`RADIAL_EDGES`).

In [ ]:
# Radial profile of the Level-2 residuals about the Level-2 centre.
radial = []
for name in names:
    r = res[name]
    rho = np.hypot(r['x'], r['y'])   # [deg] from the Level-2 centre on the tangent plane
    for comp in COMPS:
        v = r['out']['L2'][1][comp]   # Level-2 residuals [km/s]
        # Rings of RADIAL_EDGES with >= 20 stars: median and its error 1.2533 SD / sqrt(n) [km/s].
        for lo, hi in zip(RADIAL_EDGES[:-1], RADIAL_EDGES[1:]):
            k = (rho > lo) & (rho < hi) & np.isfinite(v)
            if k.sum() >= 20:
                radial.append(dict(population=name, component=comp, rho_lo=lo, rho_hi=hi, n=int(k.sum()), median=np.median(v[k]),
                                   err=1.2533 * np.std(v[k]) / np.sqrt(k.sum())))
# Table and figure: three equal panels (v_t, v_r, v_los), one line per population, offset by 0.04 deg in x per population.
radial = pd.DataFrame(radial)
radial.to_csv(ROOT / f'results/kin_residual_radial{SUFFIX}.csv', index=False, float_format='%.3f')
fig, ax = panel_grid(1, 3, width=7.0, left=0.55, right=0.1, hgap=0.5, top=0.6, aspect=0.8)
for a, comp, letter in zip(ax[0], COMPS, 'abc'):
    a.text(0.04, 0.92, f'({letter})', transform=a.transAxes, fontsize=7)
    for (name, g), colour in zip(radial[radial.component == comp].groupby('population', sort=False), COLOURS):
        a.errorbar(0.5 * (g.rho_lo + g.rho_hi) + 0.04 * (COLOURS.index(colour) - 2.5), g['median'], yerr=g.err, fmt='o-', ms=2.5,
                   lw=0.8, color=colour, label=name.replace('giants ', 'giants, '))
    a.axhline(0, color='0.6', lw=0.5)   # zero residual
    a.set(xlabel=r'$\rho$ from the Level-2 centre [deg]', ylabel=r'median $\Delta$' + LABELS[comp] + r' [km s$^{-1}$]', ylim=(-15, 15))
ax[0, 0].legend(loc='lower left', bbox_to_anchor=(0, 1.02), ncol=3, fontsize=5, frameon=False)
fig.savefig(ROOT / f'figures/paper_kin_residual_radial{SUFFIX}.png')
plt.close(fig)

### The data against position angle
Median $v_t$, $v_r$ and $v_{\rm los}$ of every group per 30-deg bin of position angle at 3-7 deg, in the joint frame.

In [ ]:
# Three stacked panels (v_t, v_r, v_los) of the 'data' model, no amplitude text.
fig, ax = panel_grid(3, 1, width=3.4, left=0.55, right=0.1, vgap=0.1, top=0.65, aspect=0.55)
profiles(ax[:, 0], 'data', 'abc', harmonics_text=False)
# y ranges [km/s] per component; x tick labels only on the bottom panel.
for i, (comp, lim) in enumerate(zip(COMPS, ((-110, 10), (-40, 40), (-60, 60)))):
    ax[i, 0].set(ylim=lim, ylabel=LABELS[comp] + r' [km s$^{-1}$]')
    if i < 2:
        ax[i, 0].tick_params(labelbottom=False)
ax[2, 0].set_xlabel('position angle [deg]')
# Legend above the top panel; save.
ax[0, 0].legend(loc='lower left', bbox_to_anchor=(0, 1.02), ncol=2, fontsize=5, frameon=False)
fig.savefig(ROOT / f'figures/paper_kin_data_profiles{SUFFIX}.png')
plt.close(fig)

### Residuals of the three models against position angle
Columns: the vdMK14 frame, the joint Level-1 fit and the joint Level-2 fit.

In [ ]:
# 3 x 3 panels: rows v_t, v_r, v_los; columns the three models, each with the amplitude range of the giants.
fig, ax = panel_grid(3, 3, width=7.0, left=0.55, right=0.1, hgap=0.1, vgap=0.1, top=0.75, aspect=0.7)
for j, model in enumerate(('vdMK14', 'L1', 'L2')):
    profiles(ax[:, j], model, 'abcdefghi'[3 * j:3 * j + 3], harmonics_text=True)
    ax[0, j].set_title(MODEL_TITLES[model], fontsize=7)
    for i, comp in enumerate(COMPS):
        # Common y range +-35 km/s; the 360 tick only on the last column.
        ax[i, j].set(ylim=(-35, 35), xticks=np.arange(0, 361 if j == 2 else 360, 60))
        # Outer tick labels only: x labels on the bottom row, y labels on the first column.
        if i < 2:
            ax[i, j].tick_params(labelbottom=False)
        else:
            ax[i, j].set_xlabel('position angle [deg]')
        if j:
            ax[i, j].tick_params(labelleft=False)
        else:
            ax[i, j].set_ylabel({'v_t': r'$\Delta v_t$', 'v_r': r'$\Delta v_r$', 'v_los': r'$\Delta v_{\rm los}$'}[comp]
                                + r' [km s$^{-1}$]')
# Legend above the top-left panel; save; print the range of the giants' amplitudes per model and component.
ax[0, 0].legend(loc='lower left', bbox_to_anchor=(0, 1.15), ncol=6, fontsize=5, frameon=False)
fig.savefig(ROOT / f'figures/paper_kin_residual_profiles{SUFFIX}.png')
print(harm[harm.population.str.startswith('giants')].groupby(['model', 'component'])[['A1', 'A2']].agg(['min', 'max']).round(1))

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_kin_maps_vt_ellipsoid.png')

**Figure 6.1.** Tangential velocity $v_t$ (km/s; 90° east of the direction away from the centre) of the
old population on the sky. Columns: giants in the five bins of measured [Fe/H], metal-poor (left) to metal-rich, and the RR Lyrae
(right). Rows: (top) data: proper motion minus the bulk motion of the LMC at the star's position, converted to km/s at the distance
of the disc plane; (middle) the joint Level-2 model with the velocity ellipsoid (model section); (bottom) significance of the
residual (data minus model) in each cell, $z=\sqrt{n}\,{\rm median}(\Delta v_i/\tilde\sigma_i)/\sqrt{\pi/2}$ for the $n$ stars of a
cell, where $\Delta v_i$ is the residual of star $i$ and $\tilde\sigma_i$ the scatter the model predicts for it in this component:
its velocity ellipsoid projected on the component plus the scaled measurement error. If the model is right, $z$ follows
approximately a unit normal distribution in every cell (the scatter $\tilde\sigma_i$ omits the outlier component, and stars with
residuals beyond 150 km/s, plus three times their own error for the proper-motion components, are dropped). Colour
scales: right of the last column (top bar for the data and model rows, bottom bar for $z$). Centre, systemic motion and disc plane of
the joint fit (Table 6.2); the RR Lyrae rotation curve, ellipsoid, error scale $k$ and outlier fraction $f$ are refitted in
that frame at 4° < ρ < 9°, so inside 4°, where Gaia underestimates its
errors more, their $|z|$ is larger. Median per 1° cell with at least 5 stars, at
2° < ρ < 9° from the vdMK14 centre. Axes: offsets east (increasing to the left) and north (deg)
of the fitted centre, (α, δ) = (81.23°, -69.61°); +: the vdMK14 centre, the origin of ρ used to select the
stars; dashed line: the fitted line of nodes. `scripts/paper_kin_residuals.py ellipsoid`

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_kin_maps_vr_ellipsoid.png')

**Figure 6.2.** As Figure 6.1 for the radial component $v_r$ of the velocity
across the sky (km/s, positive away from the centre). `scripts/paper_kin_residuals.py ellipsoid`

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_kin_maps_vlos_ellipsoid.png')

**Figure 6.3.** As Figure 6.1 for the line-of-sight velocity $v_{\rm los}$ (km/s,
minus the bulk motion of the LMC along the star's line of sight), giants only (the RR Lyrae have no line-of-sight velocities).
`scripts/paper_kin_residuals.py ellipsoid`

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_kin_data_profiles_ellipsoid.png')

**Figure 6.4.** The data of Figure 6.1–Figure 6.3
against position angle (deg, north through east, about the fitted centre) at 3° < ρ < 7°:
(a) $v_t$, (b) $v_r$, (c) $v_{\rm los}$ (km/s), median per 30° bin for each group (colours, legend above: giants
from red, metal-poor, to blue, metal-rich; purple, RR Lyrae). Dotted vertical lines: the line of nodes; dash-dotted: the minor axis.
`scripts/paper_kin_residuals.py ellipsoid`

## Disc model and fits

### Overview

We describe each group of stars as a thin, rotating disc seen in perspective. The model has three parts. (i) A geometry: where the
disc is, how it is oriented, and where on it each star lies. (ii) A mean velocity field: the bulk motion of the LMC through space
plus a rotation curve and an oval ($m=2$) distortion of the flow in the disc plane. (iii) The random motions about that mean: a
Gaussian velocity ellipsoid with three different dispersions along the radial, azimuthal and vertical directions of the disc. Each
star's proper motion and line-of-sight velocity are compared with the projection of this model onto its own position on the sky,
with its own measurement errors. The model is kinematic: it describes the velocities and does not use a gravitational potential.
The assumptions are listed at the end of this section, and Table 6.1 lists every parameter.

### Geometry

The stars are placed on a thin disc whose centre lies at the fixed distance $D_0$ = 50.1 kpc (as in vdMK14; the
stellar-mass chapter uses the distance modulus 18.477, 49.6 kpc, which would scale velocities and
radii by 1 per cent) in the direction of the unit vector $\hat{\boldsymbol r}_0$, offset by $(\Delta x,\Delta y)$
degrees east and north of the vdMK14 centre. The plane has unit normal $\hat{\boldsymbol n}$, set by the inclination $i$ (the angle
between the normal and the line of sight; $i=0$ is face-on) and the position angle $\theta$ (north through east) of the line of
nodes, whose unit vector $\hat{\boldsymbol l}$ points to position angle $\theta$. A star seen in the direction of the unit vector
$\hat{\boldsymbol r}$ is placed where its line of sight meets the plane,

$$
D=D_0\,\frac{\hat{\boldsymbol n}\cdot\hat{\boldsymbol r}_0}{\hat{\boldsymbol n}\cdot\hat{\boldsymbol r}},\qquad
\boldsymbol x=D\hat{\boldsymbol r}-D_0\hat{\boldsymbol r}_0,\qquad R=|\boldsymbol x|,
$$

so each star has a distance $D$, a position $\boldsymbol x$ in the plane and a radius $R$ from the fitted centre. Its in-plane unit
vectors are $\hat{\boldsymbol e}_R=\boldsymbol x/R$ (radial) and $\hat{\boldsymbol e}_\phi=\hat{\boldsymbol n}\times\hat{\boldsymbol
e}_R$ (azimuthal), and its azimuth $\phi$ in the disc plane is measured from the line of nodes ($\cos\phi=\hat{\boldsymbol
e}_R\cdot\hat{\boldsymbol l}$) and increases along $\hat{\boldsymbol e}_\phi$.

### Mean velocity

The mean velocity of the stars at $(R,\phi)$ is the systemic velocity of the centre plus motion in the disc plane,

$$
\boldsymbol V=v_{{\rm los},0}\,\hat{\boldsymbol r}_0+\kappa D_0\,(\mu_{\alpha*,0}\,\hat{\boldsymbol e}_{E,0}+\mu_{\delta,0}\,
\hat{\boldsymbol e}_{N,0})+v_\phi\,\hat{\boldsymbol e}_\phi+v_R\,\hat{\boldsymbol e}_R,
$$

where $(\mu_{\alpha*,0},\mu_{\delta,0})$ is the proper motion of the centre (east, $\mu_{\alpha*}=\mu_\alpha\cos\delta$, and north;
mas/yr) and $v_{{\rm los},0}$ its line-of-sight velocity (km/s), $\hat{\boldsymbol e}_{E,0}$ and $\hat{\boldsymbol e}_{N,0}$ point east
and north at the centre, $v_\phi$ and $v_R$ are the mean azimuthal and radial velocities in the plane, and $\kappa$ =
4.74047 km/s per (mas/yr × kpc) converts a proper motion at a distance into a velocity. The model proper motion
$\boldsymbol\mu_{\rm m}$ and line-of-sight velocity $v_{\rm m}$ of a star are the projections of this one vector on the star's own
east, north and line-of-sight unit vectors $\hat{\boldsymbol e}_E$, $\hat{\boldsymbol e}_N$, $\hat{\boldsymbol r}$,

$$
\boldsymbol\mu_{\rm m}=\frac{1}{\kappa D}\bigl(\boldsymbol V\cdot\hat{\boldsymbol e}_E,\ \boldsymbol V\cdot\hat{\boldsymbol e}_N\bigr),\qquad
v_{\rm m}=\boldsymbol V\cdot\hat{\boldsymbol r},
$$

so the perspective effects of the bulk motion across the 18° of sky the samples span, which reach tens of km/s, are
included exactly rather than removed beforehand. The rotation curve is

$$
v_{\rm rot}(R)=v_0\tanh(R/R_0)\,[1+s\,(R-R_\sigma)],\qquad R_\sigma=4\ {\rm kpc}:
$$

a rise over a radial scale $R_0$ towards a level $v_0$, multiplied by a linear factor with slope $s$ (per kpc) that lets the curve
flatten or decline in the outer disc; $R_\sigma$ = 4 kpc is a fixed reference radius, the middle of the fitted radii.
In the symmetric disc (Level 1) the stars move on circles, $v_\phi=v_{\rm rot}(R)$ and $v_R=0$; we fitted it only with $s=0$ and the
sky-aligned dispersions of the checks below, as a reference for the residuals. Level 2, our default, adds in-plane non-circular
motion with an $m=2$ pattern, the shape of the flow in a bar-like or tidally stretched potential:

$$
\begin{aligned}
v_\phi&=v_{\rm rot}(R)\,[1+m_{2c}\cos2\phi+m_{2s}\sin2\phi],\\
v_R&=v_{\rm rot}(R)\,[v_{R,0}+r_{2c}\cos2\phi+r_{2s}\sin2\phi].
\end{aligned}
$$

The five streaming coefficients $m_{2c}$, $m_{2s}$, $v_{R,0}$, $r_{2c}$ and $r_{2s}$ are dimensionless fractions of the local
rotation speed. $m_{2c}<0$ slows the rotation along the line of nodes and speeds it up along the minor axis; $r_{2c}$ and $r_{2s}$
describe inward and outward motion with the same two-fold pattern; $v_{R,0}$ is a uniform expansion ($>0$) or contraction. Averaged
over azimuth the $m=2$ terms vanish, so $v_{\rm rot}(R)$ is the mean rotation of the axisymmetric part of the flow.

### The rotation speed we quote

The three parameters of the rotation curve are strongly correlated: a larger $v_0$ with a larger $R_0$ and a more negative $s$ gives
nearly the same curve over the radii the stars cover. In the giant groups $v_0$ ranges from 67 to
110 km/s and $R_0$ from 3.2 to 5.0 kpc, and with $s<0$ the curves peak at
$R$ = 4.6–5.0 kpc, so $v_0$ is a level the stars never reach. We therefore quote the speed at
$R_\sigma$, where the linear factor equals one,

$$
v_{\rm rot}(4\,{\rm kpc})=v_0\tanh(4\,{\rm kpc}/R_0),
$$

computed for every posterior draw; its posterior half-width (half the range between the 16th and 84th percentiles) is
0.2–1.2 km/s for the giants. All stars at 2° < ρ < 9°,
not only those near 4 kpc, constrain it through the shape of the curve. $v_{\rm rot}$ is the mean rotation of the
stars, not the circular speed of the potential: the mean rotation of a population lags the circular speed by its asymmetric drift,
which grows with its dispersion and which we do not model (it is the subject of the page on rotation and the circular speed).

### Velocity ellipsoid

The velocities of the stars scatter about $\boldsymbol V$ with a Gaussian distribution whose covariance, in the disc frame
$(\hat{\boldsymbol e}_R,\hat{\boldsymbol e}_\phi,\hat{\boldsymbol n})$ at the star's position, is

$$
\Sigma_{\rm disc}(R)={\rm diag}(\sigma_R^2,\sigma_\phi^2,\sigma_z^2)\,e^{2\gamma(R-R_\sigma)}.
$$

$\sigma_R$, $\sigma_\phi$ and $\sigma_z$ are the radial, azimuthal and vertical velocity dispersions at $R_\sigma$, and every
dispersion changes with radius by the same factor $e^{\gamma(R-R_\sigma)}$, with $\gamma={\rm d}\ln\sigma/{\rm d}R$ (per kpc): the
ellipsoid keeps its shape and its size changes by the factor $e^{\gamma}$ per kpc. The ellipsoid is aligned with the disc: its axes
point along $\hat{\boldsymbol e}_R$, $\hat{\boldsymbol e}_\phi$ and $\hat{\boldsymbol n}$, with no vertex deviation (a rotation of the
ellipsoid in the plane) and no tilt. To compare it with the data we rotate it onto each star's sky directions. The matrix $P_i$ with
elements $(P_i)_{ab}=\hat{\boldsymbol u}_a\cdot\hat{\boldsymbol d}_b$, where $\hat{\boldsymbol u}_a$ runs over star $i$'s (east, north,
line-of-sight) unit vectors and $\hat{\boldsymbol d}_b$ over $(\hat{\boldsymbol e}_R,\hat{\boldsymbol e}_\phi,\hat{\boldsymbol n})$ at
its position, does this. We write the data of star $i$ as one vector in km/s, $\boldsymbol y_i=(\kappa D_i\mu_{\alpha*},\kappa
D_i\mu_\delta,v_{\rm los})$, with model $\boldsymbol y_{\rm m}=(\kappa D_i\boldsymbol\mu_{\rm m},v_{\rm m})$ and covariance

$$
\Sigma_i=P_i\,\Sigma_{\rm disc}(R_i)\,P_i^{\rm T}+{\rm blockdiag}\bigl(k^2\kappa^2D_i^2C_i,\ e_i^2\bigr),
$$

the intrinsic scatter projected on the sky plus the measurement errors ($C_i$, $k$ and $e_i$ are defined below). Proper motions and
line-of-sight velocity are thus described by one tensor, and their covariance, of order $\sin i\cos i\,(\sigma_z^2-\sigma_{R,\phi}^2)$,
is kept.

### Measurement errors and contamination

$C_i$ is the 2 × 2 Gaia covariance matrix of the star's proper motion, multiplied by $k^2$: Gaia underestimates its proper-motion
errors, more so in crowded fields, and each group has one free error scale $k$. The fit separates $k$ from the dispersions because
the measurement term grows with each star's own error, while the intrinsic term is the same for bright and faint stars at one
position; the spread of errors within a group therefore fixes both. The errors span a wider range for the giants than for the RR
Lyrae, whose $k$ and dispersions are therefore much more strongly correlated: between the 5th and 95th percentiles the errors (the
mean of the two components, in km/s at the LMC distance) span a factor of 2.5–3.5 in each
giant bin (6–21 km/s over all bins) but only 1.7 for the RR Lyrae at
4° < ρ < 9° (47–80 km/s). $k$ also takes any excess proper-motion scatter that scales
with the quoted errors. The fitted values are 1.28–1.67 for the giants and 1.13 for the
RR Lyrae at 4° < ρ < 9° (chapter 7); fitted at 2° < ρ <
9° the RR Lyrae need $k$ = 1.27, because their crowded inner stars have errors underestimated more.
$e_i$ is the quoted error of the line-of-sight velocity, used as it is. A broad component with fraction $f$ takes the Milky Way stars
and gross errors that remain after the cuts of the data chapters: its velocities follow the same mean with an extra width $S$ =
1 mas/yr (about 240 km/s) in every component. The likelihood of star $i$ is

$$
\mathcal{L}_i=(\kappa D_i)^2\bigl[(1-f)\,\mathcal{N}(\boldsymbol y_i\,|\,\boldsymbol y_{\rm m},\Sigma_i)
+f\,\mathcal{N}(\boldsymbol y_i\,|\,\boldsymbol y_{\rm m},\Sigma_i+\kappa^2D_i^2S^2I)\bigr],
$$

where $\mathcal{N}(\boldsymbol y\,|\,\boldsymbol m,\Sigma)$ is the Gaussian density of $\boldsymbol y$ with mean $\boldsymbol m$ and
covariance $\Sigma$, $I$ the identity matrix, and the factor $(\kappa D_i)^2$ turns the density in km/s into a density in proper
motion, the measured quantity. A star without a line-of-sight velocity (all RR Lyrae) enters with the proper-motion block of
$\boldsymbol y_i$ and $\Sigma_i$ only, which is the Gaussian marginalised over the missing component. The fits are unweighted: the
selection functions of the data chapters change how many stars are observed at each position, not their velocities, so they drop out
of a likelihood conditional on the positions, provided that the selection does not depend on velocity within the cuts of the data
chapters. Priors are uniform within wide bounds (Table 6.1): $|\Delta x|,|\Delta y|\leq$ 3°;
5° ≤ $i$ ≤ 80°; systemic proper motion within 2 mas/yr and $v_{{\rm los},0}$
within 30 km/s of the vdMK14 values; 0 ≤ $v_0$ ≤ 200 km/s; 0.1 ≤ $R_0$ ≤
10 kpc; $|s|\leq$ 0.3 per kpc; streaming coefficients within ±0.5;
3 ≤ $\sigma_{R,\phi,z}$ ≤ 150 km/s; $|\gamma|\leq$ 1 per kpc;
0.5 ≤ $k$ ≤ 3; ${\rm logit}\,f=\ln[f/(1-f)]$ uniform between -10 and
0.

### Free parameters

A group fitted on its own has 21 free parameters: the centre ($\Delta x,\Delta y$), the systemic motion
($\mu_{\alpha*,0},\mu_{\delta,0},v_{{\rm los},0}$), the orientation ($i,\theta$), the rotation curve ($v_0,R_0,s$), the five streaming
coefficients, the ellipsoid ($\sigma_R,\sigma_\phi,\sigma_z,\gamma$), $k$ and $f$. The RR Lyrae have 20: their
proper motions depend on $v_{{\rm los},0}$ only through the perspective terms, too weakly to fit it, and we hold it at
261.1 km/s (vdMK14). Table 6.1 lists them with their meaning, units and priors.

| Symbol | Meaning | Unit | Prior (uniform) | Joint fit |
|---|---|---|---|---|
| $\Delta x$, $\Delta y$ | position of the disc centre, east and north of the vdMK14 centre | deg | ±3 | shared |
| $\mu_{\alpha*,0}$, $\mu_{\delta,0}$ | proper motion of the centre | mas/yr | ±2 about vdMK14 | shared |
| $v_{{\rm los},0}$ | line-of-sight velocity of the centre | km/s | ±30 about vdMK14 | shared |
| $i$ | inclination of the disc plane | deg | 5–80 | shared |
| $\theta$ | position angle of the line of nodes, north through east | deg | 0–360 | shared |
| $v_0$ | level of the tanh rise of the rotation curve | km/s | 0–200 | own |
| $R_0$ | radial scale of the rise | kpc | 0.1–10 | own |
| $s$ | outer slope of the rotation curve (linear factor about $R_\sigma$) | 1/kpc | ±0.3 | own |
| $m_{2c}$, $m_{2s}$ | $m=2$ modulation of the rotation, fraction of $v_{\rm rot}$ | – | ±0.5 | shared |
| $v_{R,0}$ | uniform radial motion (expansion > 0), fraction of $v_{\rm rot}$ | – | ±0.5 | shared |
| $r_{2c}$, $r_{2s}$ | $m=2$ radial motion, fraction of $v_{\rm rot}$ | – | ±0.5 | shared |
| $\sigma_R$, $\sigma_\phi$, $\sigma_z$ | radial, azimuthal and vertical dispersions at $R_\sigma$ | km/s | 3–150 | own |
| $\gamma$ | ${\rm d}\ln\sigma/{\rm d}R$, common to the three dispersions | 1/kpc | ±1 | own |
| $k$ | scale of the Gaia proper-motion errors | – | 0.5–3 | own |
| $f$ | fraction of stars in the broad (contaminant) component | – | logit $f$ in [-10, 0] | own |
| $v_{\rm rot}(4\,{\rm kpc})$ | derived: rotation speed at $R_\sigma$, $v_0\tanh(R_\sigma/R_0)$ | km/s | – | own |

**Table 6.1.** Parameters of the kinematic model. Prior: uniform within the range given. Joint fit: shared by all groups
in stage 2, or each group's own. A group fitted on its own (stage 1) has all 21 parameters free; the RR Lyrae have
20 (no line-of-sight velocities: $v_{{\rm los},0}$ held at 261.1 km/s). Fixed: the distance of
the centre $D_0$ = 50.1 kpc, the reference radius $R_\sigma$ = 4 kpc and the extra width of the
outlier component $S$ = 1 mas/yr. The vdMK14 centre and systemic motion serve only as the origin of the offsets and
of the prior ranges (`lmcold.kinfit.BOUNDS`).

### What constrains the three dispersions

At inclination $i$ the line of sight sees, at azimuth $\phi$, the dispersion

$$
\sigma_{\rm los}^2(\phi)=\sigma_z^2\cos^2i+\sin^2i\,(\sigma_R^2\sin^2\phi+\sigma_\phi^2\cos^2\phi),
$$

mostly $\sigma_z$ ($\cos^2i$ = 0.78 at $i$ = 28°). The proper motions see mostly the in-plane
ellipsoid, with $\sigma_z$ entering only through $\sin^2i$ = 0.22: for a star on the line of nodes they measure
$\sigma_R$ along the direction to the centre and $\sigma_\phi$ across it, and the inclination mixes the two elsewhere. With proper
motions and line-of-sight velocities the three dispersions are therefore constrained separately; with proper motions alone (the RR
Lyrae) $\sigma_R$ and $\sigma_\phi$ are constrained but $\sigma_z$ is not. For the giants, the proper-motion errors $\epsilon$ (median
12–15 km/s per component in the five bins), scaled by $k$, are
17–22 km/s, comparable to the dispersions of the metal-rich bins: a change of $k$ by
0.1 moves the $\sigma_\phi$ of the most metal-rich bin by about 2 km/s
($2k\epsilon^2\Delta k\approx$ 55 km²/s² against $\sigma_\phi^2\approx$ 300 km²/s²), so one $k$
per group is an assumption that matters, which chapter 7 tests. The RR Lyrae errors (median 64 km/s per
component) are about twice their dispersions, so the scatter fixes mainly $\sigma_{R,\phi}^2+k^2\epsilon^2$ and one $k$ must describe
all stars of the sample (chapter 7). The dispersions are averages over the part of the disc a group covers, with the radial dependence of the
ellipsoid above. Mean motions that the model does not contain, such as the steeper inner rise of the rotation curve and the radial
flow found in the residuals below, add to them or remain in the residuals.

### Tests

We tested the model on mocks drawn from physical discs (positions from an exponential disc in the footprint, Gaia errors of the
nearest RR Lyrae; see the checks, Figure 6.7). With proper motions and line-of-sight velocities it recovers
$\sigma_R$ and $\sigma_z$ to within 1 km/s and $\sigma_\phi$ to within 1.3 km/s for cold and hot,
thin and thick discs, including three whose mean rotation lags by the asymmetric drift and is therefore not of the tanh form above;
with a pure tanh curve (outer slope $s=0$) the dispersions change by less than 0.5 km/s. With proper motions alone it
recovers $\sigma_R$ and $\sigma_\phi$ to within 2.5 km/s (the thick disc: $\sigma_\phi$ 4.8
km/s too high) but overestimates $\sigma_z$ by 5–7 km/s, with errors of
4.2–5.5 km/s. The model replaces the sky-aligned dispersions of our first fits (one dispersion for
both proper-motion directions and one along the line of sight; see the checks). It fits the data better in every group: the
log-likelihood $\ln L$ of the stars is higher by $\Delta\ln L$ = 37–1 443 for the giants at
2° < ρ < 9° and 26 for the RR Lyrae at 4° < ρ <
9°, with 3 more parameters (4 for the RR Lyrae).

### Assumptions

In summary, the model assumes that: (1) the stars of a group lie in one thin plane, without thickness or spread along the line of
sight; (2) the bulk motion of the LMC is one velocity vector; (3) the mean motion in the plane is a rotation curve of the form above
with an $m=2$ distortion, and has no lopsided ($m=1$) part, no warp or change of orientation with radius or time, and no mean vertical
motion (tilted, scaled and turning discs are tested as alternatives in chapter 7); (4) the random motions are Gaussian, with an
ellipsoid aligned with the disc whose three axes change with radius by one common factor; (5) one scale $k$ corrects the Gaia errors
of all stars of a group; (6) the remaining contaminants form one broad component; (7) the selection of the stars does not depend on
their velocities. No dynamical relation links the rotation curve to the ellipsoid (no Jeans equation, the moment of the collisionless Boltzmann
equation that ties mean motions and dispersions to the potential; no asymmetric-drift or epicyclic relation), so the model measures how the stars move, not the mass that makes them move.

### Two stages

Levels 1 and 2 describe the mean motion; the two stages describe how the groups are combined. We fit the model in two ways, which
make two different assumptions. In stage 1 each group is fitted on its own, with its own centre, systemic motion, orientation and
streaming. In stage 2 all groups are fitted together and share these (12 parameters), each keeping its own
rotation curve, ellipsoid, $k$ and $f$. The two stages answer two different questions, and we use each for one of them. The rotation
and dispersions of a group (chapter 7) come from stage 1, which does not force one geometry on populations that may not share it.
The maps and profiles (Figure 6.1–Figure 6.4 and the right column of
Figure 6.5) and the geometry (Table 6.2) come from stage 2, which puts all groups in one
frame. For the RR Lyrae in the maps we keep the stage-2 frame and refit their own parameters at 4° < ρ <
9° (maximum likelihood), as for the values we quote. Chapter 7 compares the two stages group by group. Imposing
the shared geometry changes the giants' $v_{\rm rot}(4\,{\rm kpc})$ by at most 0.7 km/s and their dispersions by
at most 0.6 km/s; for the RR Lyrae, in the joint fit of all groups at 4° < ρ <
9° ($i$ = 28.6°, $\theta$ = 145.5°), it changes $v_{\rm rot}(4\,{\rm kpc})$ by
0.3 km/s and $\sigma_R$ and $\sigma_\phi$ by 3.1 and 3.2 km/s, about one
standard deviation (stage-1 posterior half-widths 3.0 and 4.4 km/s).
Table 6.2 lists the stage-2 parameters with the range of the stage-1 values. The giant groups agree on the line of nodes to
within 2.5° but less well on $i$ (24.6–28.3°) and on the centre
(0.32–0.54° east): the stage-2 errors are conditional on a shared geometry, and the stage-1 range is
the better measure of the uncertainty of each geometric parameter. The centre offset and $\mu_{\delta,0}$ are strongly correlated in
every group (shown for the sky-aligned fits in Figure 6.13).

### Posteriors

We sample the posteriors with the No-U-Turn Sampler (NUTS; [Hoffman & Gelman 2014](https://jmlr.org/papers/v15/hoffman14a.html)), a
Hamiltonian Monte Carlo method, in [NumPyro](https://doi.org/10.48550/arXiv.1912.11554) ([Phan et al. 2019](https://ui.adsabs.harvard.edu/abs/2019arXiv191211554P)),
4 chains of 1 000 draws after 1 500 warm-up steps (500 for the
RR Lyrae fit at 4° < ρ < 9°). Every run has a Gelman–Rubin statistic $\hat R$ (the ratio
of the spread between chains to that within them, 1 for converged chains) ≤ 1.017 and at least 217
effective (independent-equivalent) draws per parameter; one stage-1 fit (giants, -1.5 to
-1.1) has 5 divergent transitions (sampler steps that failed to follow the posterior) among its
4 000 draws, the others none. Each optional term of the model was added with a test that recovers it from a mock
(`tests/test_kinfit.py`).

In [ ]:
import pandas as pd
# Posterior summaries: level 1 (centre fit), level 2 (ellipsoid fit, giants) and level 2 'outer' run (RR Lyrae, 4 < rho < 9 deg)
L1 = pd.read_csv('results/kin_mcmc_data_centre.csv')
L2 = pd.read_csv('results/kin_mcmc_data_ellipsoid.csv')
RR = pd.read_csv('results/kin_mcmc_data_ellipsoid_outer.csv')
# Shared parameter: (table label, decimals of the shared value, decimals of the stage-1 range)
LABEL = {'dx': ('Δx (deg)', 3, 2), 'dy': ('Δy (deg)', 3, 2), 'pmra0': ('μ_α*,0 (mas/yr)', 4, 3),
         'pmdec0': ('μ_δ,0 (mas/yr)', 3, 3), 'vlos0': ('v_los,0 (km/s)', 1, 1), 'inc': ('i (deg)', 1, 1), 'theta': ('θ (deg)', 1, 1),
         'm2c': ('m_2c', 3, 3), 'm2s': ('m_2s', 3, 3), 'vr0': ('v_R,0', 3, 3), 'r2c': ('r_2c', 3, 3), 'r2s': ('r_2s', 3, 3)}
STREAM = ('m2c', 'm2s', 'vr0', 'r2c', 'r2s')   # streaming coefficients: no stage-1 range in the table


def shared(t, p, d):
    """Stage-2 shared value: median ± half the 16th-84th percentile range, or '–' if the model has no such parameter."""
    r = t[(t.stage == 2) & (t.population == 'shared') & (t.param == p)]
    return f"{r['median'].iloc[0]:.{d}f} ± {(r.p84.iloc[0] - r.p16.iloc[0]) / 2:.{d}f}" if len(r) else '–'


rows = []
# Stage-1 column: range of the per-bin giant medians, the RR Lyrae value in brackets
for p, (label, d, d1) in LABEL.items():
    g = L2[(L2.stage == 1) & L2.population.str.startswith('giants') & (L2.param == p)]['median']
    r = RR[(RR.stage == 1) & (RR.population == 'RR Lyrae') & (RR.param == p)]['median']
    stage1 = '' if p in STREAM else f'{g.min():.{d1}f} to {g.max():.{d1}f}' + (f' ({r.iloc[0]:.{d1}f})' if len(r) else '')
    rows.append((label, shared(L1, p, d), shared(L2, p, d), stage1))
pd.DataFrame(rows, columns=['Parameter', 'Level 1', 'Level 2', 'Level 2, stage 1']).set_index('Parameter')

**Table 6.2.** Shared parameters of the joint (stage-2) fits: posterior medians ± half the 16th–84th percentile range.
Level 1: symmetric disc with sky-aligned dispersions (checks); Level 2: the model of this section with the velocity ellipsoid. Last
column: range of the stage-1 Level-2 values of the five giant groups, each fitted on its own (RR Lyrae in brackets, fitted at
4° < ρ < 9°). $\Delta x$, $\Delta y$: offset east and north of the vdMK14 centre,
(α, δ) = (79.88°, -69.59°). Streaming terms in units of $v_{\rm rot}(R)$. From
`results/kin_mcmc_data_centre.csv` (Level 1), `kin_mcmc_data_ellipsoid.csv` and `kin_mcmc_data_ellipsoid_outer.csv`, written by
`scripts/fit_kinematics_mcmc.py`.

### Running the fit

`scripts/fit_kinematics_mcmc.py` performs every fit of this section. For one model (the argument `ellipsoid` for the model above,
`stream` for the sky-aligned model of the checks, `centre` for Level 1) it reads the giants and the RR Lyrae, keeps the stars in the
radial range of the fit, and finds the maximum-likelihood parameters of each group. From there NUTS samples the posterior of each
group on its own (stage 1) and of all groups together with a shared geometry (stage 2). The likelihood is the one written above: for
each star, the model velocity projected on its own sky directions, compared with its proper motion and line-of-sight velocity
through a Gaussian whose covariance is the projected velocity ellipsoid plus the scaled measurement errors, mixed with the broad
outlier component.

The script writes the summary tables `results/kin_mcmc_data_<model>.csv` (median, 16th and 84th percentiles and convergence
diagnostics of every parameter), which the tables and macros of this page read, and the posterior draws
`data/kin_mcmc_data_<model>_samples.npz`. The draws of the runs the released scripts read (the velocity-ellipsoid and sky-aligned
models, each over the full radial range and beyond 4°) are in Zenodo bundle 1, so the corner plots, the
comparison of the two models and the derived rotation speeds are made from them without rerunning the fit.

<details><summary>About <code>scripts/fit_kinematics_mcmc.py</code></summary>

```text
The kinematic fit: posterior of the LMC centre, bulk motion, disc orientation, rotation curve, streaming and velocity dispersions
of each old population, from stellar proper motions and line-of-sight velocities (lmcold.kinfit.sample, NUTS). Its result tables
give the rotation, dispersions and geometry quoted in the paper (kinematics section and its appendix; KB chapters 6 and 7), and the
centre and disc plane used by Letter Fig. 1 (scripts/paper_letter_signal.py) and by the maps of scripts/paper_kin_residuals.py.

The model in plain words (lmcold.kinfit; equations in KB chapter 6, 'Disc model and fits'). Each population is a thin disc seen in
perspective, its centre at the fixed distance D0 (lmcold.constants.LMC) and offset (dx, dy) deg from the adopted (vdMK14) centre,
its plane set by the inclination inc and the line of nodes theta. Each star is placed where its line of sight meets that plane.
The mean velocity there is the bulk motion of the LMC (systemic proper motion pmra0, pmdec0 and line-of-sight velocity vlos0 of the
centre) plus rotation in the plane, v_rot(R) = v0 tanh(R / R0) [1 + vslope (R - kinfit.R_SIG)], and, where the model frees them,
the m = 2 streaming terms (m2c, m2s, vr0, r2c, r2s: fractions of v_rot) or a lopsided (m = 1) part (m1c, m1s). This one velocity
vector is projected on the star's own east, north and line-of-sight directions, so the perspective effects of the bulk motion are
part of the model. The stars scatter about the mean with a Gaussian distribution: in the 'ellipsoid' model a velocity ellipsoid
aligned with the disc (sigma_R, sigma_phi, sigma_z [km/s] at R_SIG, all three scaled by exp(dlnsig_dR (R - R_SIG))); in the other
models one dispersion sigma in both proper-motion directions and sigma_los along the line of sight (the 'sky-aligned' model).
The likelihood of a star is a mixture of two Gaussians in (proper motion, line-of-sight velocity): with weight 1 - f, covariance =
intrinsic scatter + the Gaia proper-motion covariance scaled by k^2 + the squared line-of-sight error; with weight f, the same
plus an extra width kinfit.S_OUT (mas/yr) in every component, for Milky Way stars and gross errors (f = 1 / (1 + exp(-logit_fout))).
A star without a line-of-sight velocity (all RR Lyrae) enters with its proper-motion part only. The fit is unweighted (the
selection depends on position, not velocity). Priors: uniform within kinfit.BOUNDS. No gravitational potential is used.

Populations: the giants in five [Fe/H] bins (proper motions and line-of-sight velocities) and the RR Lyrae members (proper motions
only) of lmcold.samples, stars at RING[0] < rho < RING[1] deg from the adopted centre. Models (second argument): 'centre' (centre
free, not lopsided), 'lopsided' (adopted centre, lopsided part m1c, m1s free), 'both', or 'stream' (centre and in-plane streaming
kinfit.STREAM free, not lopsided: the sky-aligned model of the KB checks), or 'ellipsoid' (as 'stream', with the disc-frame
velocity ellipsoid sigma_R, sigma_phi, sigma_z and its radial gradient dlnsig_dR in place of sigma and sigma_los: kinfit's ellipsoid
likelihood; and the outer slope vslope of the rotation curve free: in the mocks of scripts/paper_kin_ellipsoid_mock.py, rotation
curves lagging a tanh by the asymmetric drift, the dispersions with vslope free and with vslope = 0 agree to 0.5 km/s). 'ellipsoid'
is the paper's model; 'centre' is Level 1 of the KB. Stage 1: each population on its own.
Stage 2: all populations together, sharing the centre, bulk motion, line-of-sight systemic velocity, inclination, line of nodes,
lopsided part and streaming, each with its own rotation curve, dispersions, error scale and outlier fraction. Both stages start from the
maximum-likelihood fits (lmcold.kinfit.fit), the shared part from the first population. D0 is fixed.
First argument 'mock': the same positions and errors, velocities drawn from the model with one shared truth (centre offset
MOCK_OFFSET, not lopsided, the other shared parameters, streaming included, from the fit to the first population) and each population's own fitted
rotation and dispersions, k = 1, no outliers. 'data': the observed velocities.
Optional third argument 'dist': the RR Lyrae also bring their relative distance moduli (OGLE Wesenheit; lmcold.samples.rrab), which
tie the disc plane to the plane the RR Lyrae lie in (kinfit distance term, with their own zero point dmu0 and scatter sigma_mu);
the run name gets the suffix _dist. For a mock the moduli are drawn from the model with the population's fitted dmu0 and sigma_mu.
Optional argument 'outer': stars at OUTER_INNER < rho < RING[1] only, run name suffix _outer. The Gaia proper-motion error scale k is
larger in the crowded inner disc (1.39 at 2-4 deg for the RR Lyrae, 1.12-1.14 beyond); with one k per population the inner stars
set it, which overstates the errors of the outer stars and, for the RR Lyrae (errors ~64 km/s per component), pushes their
proper-motion dispersion to its lower bound. The proper-motion dispersions of the paper come from this run
(scripts/paper_kin_dispersion_cut.py checks the cut).

Writes results/kin_mcmc_<kind>_<model>.csv (median, 16th and 84th percentiles, r_hat, n_eff per parameter, number of divergent
transitions of the run; truth for the mock), the posterior samples to data/kin_mcmc_<kind>_<model>_samples.npz (keys
'stage|population index or all|site', each of shape (chains, draws); population names in 'names'; for
scripts/paper_kin_corner.py, scripts/kin_model_comparison.py and the KB) and figures/fig_kin_mcmc_<kind>.png (centre and lopsided
part, each model run so far).

Inputs: lmcold.samples (release_giants_apogee and release_giants_boss via samples.giants(), and data/rrab.parquet, see their sources.yaml entries). Model and likelihood:
lmcold.kinfit. The samples (data/*.npz) are large and stay out of git. The posterior samples that the released scripts read (runs
'data stream' and 'data ellipsoid', each at 2-9 deg and with 'outer'; data/sources.yaml entries release_kin_mcmc_*) are in Zenodo
bundle 1, and the result tables (results/kin_mcmc_*.csv) are in the repository, so nothing downstream needs this fit to be rerun.
Figure: one row per model; left, the centre offset (deg east, deg north; east to the left as on the sky) of each stage-1 fit
(coloured) and of the joint fit (black star) with 16th-84th percentile bars; right, the lopsided part (m1c, m1s), or for 'stream'
and 'ellipsoid' the azimuthal m = 2 part (m2c, m2s). For a mock the truth (shared values of the csv) is a black cross. Diagnostic
figure (not in the paper).
Run time: about 30 min per sky-aligned model on 14 cores with nothing else running (four chains in parallel); 'data ellipsoid' about
4 h (four chains, one thread each) and 'data ellipsoid outer' about 2.5 h on one machine. Run one model at a time.

Run: .venv/bin/python scripts/fit_kinematics_mcmc.py data both [dist] [outer]    (kind: mock | data; model: centre | lopsided | both | stream | ellipsoid)
     The paper's runs: data ellipsoid; data ellipsoid outer; data stream; data stream outer; data centre.
```
</details>

*Shown, not run here: the giant and RR Lyrae catalogues of Zenodo bundle 1; about 4 h of MCMC for the main run (data ellipsoid), about 2.5 h for its outer version. The cells below are the script itself; run it from the repository with* `python scripts/fit_kinematics_mcmc.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The sampler set-up (four chains run in parallel), the radial ranges of the stars, and the parameters each model holds fixed.

```python
import sys

import numpyro

numpyro.set_host_device_count(4)  # four CPU 'devices' so the four chains run in parallel; must precede the JAX import

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpyro.diagnostics import summary

from lmcold import kinfit, samples
from lmcold.constants import KAPPA, LMC
from lmcold.footprint import angular_distance
from lmcold.layout import panel_grid
from lmcold.sources import DATA, ROOT

RING = (2., 9.)  # deg from the adopted centre: inner edge avoids the bar region, outer edge the sparse outskirts
OUTER_INNER = 4.  # deg, inner edge of the 'outer' runs (one Gaia error scale k fits the stars beyond)
MOCK_OFFSET = (0.5, 0.0)  # deg east, north: true centre offset of the 'mock' runs
CHAINS, WARMUP, DRAWS = 4, 500, 1000  # NUTS chains, warm-up steps and kept draws per chain
# Warm-up of the main (2-9 deg) ellipsoid run: with 500 steps R0 and v0 of the -1.5 < [Fe/H] < -1.1 giants mix poorly (stage 1
# r_hat 1.08, 34 effective draws; 2026-11-05), with 1500 r_hat <= 1.008. The outer run keeps 500 (r_hat 1.016; 1.029 with 1500).
WARMUP_ELLIPSOID = 1500
# Catalogue columns passed to kinfit.data, in its argument order.
COLS = ('ra', 'dec', 'pmra', 'pmdec', 'pmra_error', 'pmdec_error', 'pmra_pmdec_corr', 'vlos', 'evlos', 'dmu')
# Parameters held at their starting values in each model (the starting values are 0: adopted centre, symmetric disc).
FIXED = {m: f + kinfit.TILT + ('qlos',) + kinfit.TURN + (() if m == 'ellipsoid' else ('vslope',)) for m, f in dict(centre=('m1c', 'm1s') + kinfit.STREAM, lopsided=('dx', 'dy') + kinfit.STREAM,
                                             both=kinfit.STREAM, stream=('m1c', 'm1s'), ellipsoid=('m1c', 'm1s')).items()}  # one plane, no LOS scale, in every model
MODELS = tuple(FIXED)  # model names in the order of the diagnostic figure's rows
```

#### Star selection and starting point
`ring` keeps the stars in the radial range of the fit. `ml` finds the maximum-likelihood parameters of one population in two
steps (geometry first with the adopted centre, then everything the model frees); NUTS starts from there.

```python
def ring(cat, inner=RING[0]):
    """Stars of catalogue dict cat (arrays) at inner < rho < RING[1] deg from the adopted LMC centre."""
    rho = angular_distance(cat['ra'], cat['dec'], LMC['ra0'], LMC['dec0'])  # [deg] from the adopted centre
    keep = (rho > inner) & (rho < RING[1])
    # The same cut applied to every column of the catalogue dict.
    return {k: np.asarray(v)[keep] for k, v in cat.items()}


def ml(d, model):
    """Maximum likelihood: centre and lopsided part fixed first, then the model's own free from there.

    Parameters
    ----------
    d : dict
        kinfit.data() arrays of one population.
    model : str
        'centre', 'lopsided', 'both', 'stream' or 'ellipsoid' (FIXED lists what each holds). 'ellipsoid' starts from the 'stream'
        fit with sigma_R = sigma_phi = sigma and sigma_z = sigma_los (sigma for proper motions only), constant with radius.

    Returns
    -------
    ndarray, shape (33,)
        Maximum-likelihood vector (kinfit.PARAMS order), used to start NUTS.
    """
    # Line-of-sight parameters are free only if the population has line-of-sight velocities (the giants).
    los = ('vlos0', 'sigma_los') if np.any(np.asarray(d['los'])) else ()
    los += ('dmu0', 'sigma_mu') if np.any(np.asarray(d['dist'])) else ()  # distance moduli: their zero point and scatter
    # Step 1: bulk motion, orientation, rotation, dispersion, error scale, outliers (kinfit.BASE without dx, dy), centre adopted.
    pf, _, _ = kinfit.fit(d, free=kinfit.BASE[2:] + los)
    # Step 2: from there, every candidate parameter except those the model holds (FIXED[model]).
    free = kinfit.BASE + los + ('m1c', 'm1s') + kinfit.STREAM
    p = kinfit.fit(d, free=tuple(n for n in free if n not in FIXED[model]), p0=pf)[0]
    if model != 'ellipsoid':
        return p
    # Ellipsoid: (sigma_R, sigma_phi, sigma_z, dlnsig_dR) start at (sigma, sigma, sigma_los or sigma, 0), then step 3 refits with
    # the ellipsoid likelihood, sigma and sigma_los replaced by the ellipsoid and vslope freed.
    g = lambda n: p[kinfit.PARAMS.index(n)]  # noqa: E731
    p[[kinfit.PARAMS.index(n) for n in kinfit.ELLIPSOID]] = [g('sigma'), g('sigma'), g('sigma_los' if los else 'sigma'), 0.0]
    free = tuple(n for n in free if n not in FIXED[model] + ('sigma', 'sigma_los')) + kinfit.ELLIPSOID + ('vslope',)
    return kinfit.fit(d, free=free, p0=p, ellipsoid=True)[0]
```

#### Posterior summary table
`rows` turns a finished NUTS run into one row per population and parameter: median, 16th and 84th percentiles, and the
convergence diagnostics (Gelman-Rubin r_hat, effective number of draws n_eff, divergent transitions).

```python
def rows(mcmc, stage, names, truth):
    """One row per (population, parameter); names maps the sample-site suffix to the population label.

    Parameters
    ----------
    mcmc : numpyro.infer.MCMC
        Finished run of kinfit.sample.
    stage : int
        1 (one population) or 2 (joint).
    names : list of str
        Population labels, indexed by the site suffix.
    truth : dict
        (population, parameter) -> true value, for mocks; empty for the data.

    Returns
    -------
    list of dict
        stage, population ('shared' for shared parameters in stage 2), param, median, p16, p84, r_hat (split Gelman-Rubin) and
        n_eff (effective sample size) from numpyro.diagnostics.summary, divergences (number of divergent NUTS transitions of the
        run, all chains), truth (NaN when not a mock).
    """
    # Convergence diagnostics per sample site (numpyro.diagnostics.summary needs the samples grouped by chain).
    s = summary(mcmc.get_samples(group_by_chain=True))
    # Divergent transitions summed over all chains and draws.
    div = int(np.sum(mcmc.get_extra_fields()['diverging']))
    out = []
    for site, x in mcmc.get_samples().items():  # x: draws of all chains pooled
        # Shared sites have no suffix; per-population sites end with '_<population index>'.
        if site in kinfit.SHARED:
            base, pop = site, 'shared' if stage == 2 else names[0]
        else:
            base, idx = site.rsplit('_', 1)
            pop = names[int(idx)]
        x = np.asarray(x)
        # Median and 16th/84th percentiles, diagnostics and the mock truth (NaN for data).
        out.append(dict(stage=stage, population=pop, param=base, median=np.median(x), p16=np.percentile(x, 16),
                        p84=np.percentile(x, 84), r_hat=float(s[site]['r_hat']), n_eff=float(s[site]['n_eff']), divergences=div,
                        truth=truth.get((pop, base), np.nan)))
    return out
```

#### The fit: load the stars, then stage 1 and stage 2
`main` reads the stars (the giant catalogues of Zenodo bundle 1 through `lmcold.samples.giants()`, the RR Lyrae members through
`lmcold.samples.rrab()`), cuts them to the radial range, finds each population's maximum-likelihood start, and samples the
posterior of each population on its own (stage 1) and of all populations together with a shared geometry (stage 2). For a
'mock' run it first replaces the velocities by draws from the model with a known truth. It writes the summary table
`results/kin_mcmc_<run>.csv` and the posterior samples `data/kin_mcmc_<run>_samples.npz`.

```python
def main(kind, model, dist=False, outer=False):
    """Stage-1 and stage-2 posteriors for 'mock' or 'data' and one model, with the RR Lyrae distance moduli if dist, stars beyond
    OUTER_INNER only if outer; writes the csv and npz of the module docstring."""
    run = f'{kind}_{model}' + ('_dist' if dist else '') + ('_outer' if outer else '')   # name of the output files
    inner = OUTER_INNER if outer else RING[0]  # inner edge of the radial range [deg]
    # Warm-up steps: WARMUP_ELLIPSOID for the main ellipsoid run, WARMUP otherwise.
    warmup = WARMUP_ELLIPSOID if model == 'ellipsoid' and not outer else WARMUP
    # Giants have no distance moduli; NaN columns make kinfit skip the terms a population lacks.
    # Load the data: (name, catalogue dict) per population, giants first (five [Fe/H] bins), RR Lyrae last.
    pops = [(f'giants {lab}', ring(cat | dict(dmu=np.full(len(cat['ra']), np.nan)), inner)) for lab, cat in samples.giants()]
    rr = samples.rrab()
    nan = np.full(len(rr['ra']), np.nan)
    # RR Lyrae have no line-of-sight velocities; their moduli are used only with dist.
    pops.append(('RR Lyrae', ring(rr | dict(vlos=nan, evlos=nan, dmu=rr['dmu'] if dist else nan), inner)))
    names = [n for n, _ in pops]
    ds = [kinfit.data(*(c[k] for k in COLS)) for _, c in pops]   # arrays kinfit needs (unit vectors, covariances) per population
    p_ml = [ml(d, model) for d in ds]   # maximum-likelihood starting vectors, shape (33,) each
    truth = {}
    if kind == 'mock':
        # One shared truth: geometry and bulk motion of the first population's fit, centre moved by MOCK_OFFSET, symmetric disc.
        shared = {n: p_ml[0][kinfit.PARAMS.index(n)] for n in kinfit.SHARED} | dict(dx=MOCK_OFFSET[0], dy=MOCK_OFFSET[1], m1c=0., m1s=0.)
        mock_ds = []
        for i, ((name, c), d, p) in enumerate(zip(pops, ds, p_ml)):
            # Truth vector q: the population's own fit with the shared values, k = 1 and no outliers (logit_fout = -8).
            q = p.copy()
            for n, v in (shared | dict(k=1., logit_fout=-8.)).items():
                q[kinfit.PARAMS.index(n)] = v
            # Model mean pmra, pmdec [mas/yr], vlos [km/s] and disc distance D [kpc] of each star under the truth.
            mra, mde, mlos, D = (np.asarray(v) for v in kinfit.model(q, d['rhat'], d['ea'], d['ed']))
            rng = np.random.default_rng(i)  # one fixed seed per population
            # Draw proper motions from the Gaia covariance plus the intrinsic dispersion sigma (q[8], km/s -> mas/yr at D), and
            # line-of-sight velocities with e_vlos and sigma_los (q[12]); only for stars that have a measured velocity.
            cov = np.asarray(d['cov']) + ((q[8] / (KAPPA * D)) ** 2)[:, None, None] * np.eye(2)
            noise = np.einsum('nij,nj->ni', np.linalg.cholesky(cov), rng.normal(size=(len(D), 2)))
            los = mlos + rng.normal(size=len(D)) * np.hypot(c['evlos'], q[12])
            # Distance moduli about dmu0 + 5 log10(D / D0) with scatter sigma_mu (q[15], q[16]), where the star has one.
            mu = q[15] + 5 * np.log10(D / LMC['D0']) + rng.normal(size=len(D)) * q[16]
            # Mock catalogue: observed positions and errors, simulated velocities and moduli, in kinfit.data layout.
            mock_ds.append(kinfit.data(*(dict(c, pmra=mra + noise[:, 0], pmdec=mde + noise[:, 1], vlos=np.where(np.isfinite(c['vlos']), los, np.nan),
                                              dmu=np.where(np.isfinite(c['dmu']), mu, np.nan))[k] for k in COLS)))
            # Truth per (population, parameter), and for the shared parameters under 'shared'.
            truth.update({(name, n): v for n, v in zip(kinfit.PARAMS, q)})
            truth.update({('shared', n): v for n, v in zip(kinfit.PARAMS, q) if n in kinfit.SHARED})
        ds = mock_ds
        # Restart from the maximum likelihood of the mock itself (as for the data), not from the truth.
        p_ml = [ml(d, model) for d in ds]
    # Stage 1: each population on its own, every parameter of the model its own.
    out, draws = [], {}
    for i, (name, d, p) in enumerate(zip(names, ds, p_ml)):
        # NUTS for one population, started at its maximum-likelihood vector; FIXED[model] parameters held.
        m = kinfit.sample([d], [p], FIXED[model], warmup, DRAWS, CHAINS, ellipsoid=model == 'ellipsoid')
        out += rows(m, 1, [name], truth)
        # Keep the samples, shape (CHAINS, DRAWS) per site, under keys '1|<population index>|<site>'.
        draws.update({f'1|{i}|{k}': np.asarray(v) for k, v in m.get_samples(group_by_chain=True).items()})
        print(f'stage 1 {name} done', flush=True)
    # Stage 2: all populations together, shared parameters started from the first population's fit.
    m = kinfit.sample(ds, p_ml, FIXED[model], warmup, DRAWS, CHAINS, ellipsoid=model == 'ellipsoid')
    out += rows(m, 2, names, truth)
    # Stage-2 samples under keys '2|all|<site>'; then save the samples and the summary table.
    draws.update({f'2|all|{k}': np.asarray(v) for k, v in m.get_samples(group_by_chain=True).items()})
    np.savez(DATA / f'kin_mcmc_{run}_samples.npz', names=names, **draws)
    t = pd.DataFrame(out)
    t.to_csv(ROOT / f'results/kin_mcmc_{run}.csv', index=False, float_format='%.5f')
    # Print the geometry and lopsided parameters, and the worst convergence diagnostics of the run.
    print(t[t.param.isin(['dx', 'dy', 'm1c', 'm1s', 'pmra0', 'pmdec0', 'inc', 'theta', 'dmu0', 'sigma_mu'])].to_string(index=False, float_format='%.3f'))
    print('max r_hat', t.r_hat.max(), 'min n_eff', t.n_eff.min(), 'divergences', t.divergences.max())
```

#### Diagnostic figure
The centre offset and the lopsided (or $m=2$) part of every stage-1 fit and of the joint fit, one row per model run so far.

```python
def figure(kind):
    """Rows: models run so far; left the centre offset, right the lopsided (or, for 'stream', m = 2) part; stage 1 by population,
    stage 2 black star."""
    plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
    # The models of this kind whose result table exists, in MODELS order: one row of panels each.
    runs = [m for m in MODELS if (ROOT / f'results/kin_mcmc_{kind}_{m}.csv').exists()]
    fig, axes = panel_grid(len(runs), 2, width=6.5, left=.6, right=2.2, hgap=.65, vgap=.6, bottom=.5, top=.3, aspect=1.)
    # One colour per stage-1 population: five giant bins (metal-poor first), then RR Lyrae.
    colours = ['#d7191c', '#fdae61', '#a6d96a', '#1a9641', '#2878b5', '#7b3294']
    for row, model in zip(axes, runs):
        t = pd.read_csv(ROOT / f'results/kin_mcmc_{kind}_{model}.csv')
        # Right panel: the m = 2 part for the models that free streaming, the lopsided part otherwise.
        m = ('m2c', 'm2s') if model in ('stream', 'ellipsoid') else ('m1c', 'm1s')
        for ax, (a, b) in zip(row, (('dx', 'dy'), m)):
            for stage in (1, 2):
                s = t[t.stage == stage]
                # Populations that have parameter a in this stage ('shared' alone in stage 2 for shared parameters).
                pops = [p for p in s.population.unique() if ((s.population == p) & (s.param == a)).any()]
                for j, pop in enumerate(pops):
                    # Median with 16th-84th percentile error bars in both coordinates.
                    x, y = (s[(s.population == pop) & (s.param == c)].iloc[0] for c in (a, b))
                    ax.errorbar(x['median'], y['median'], [[y['median'] - y.p16], [y.p84 - y['median']]],
                                [[x['median'] - x.p16], [x.p84 - x['median']]], fmt='*' if stage == 2 else 'o',
                                ms=10 if stage == 2 else 4, lw=.8, color='k' if stage == 2 else colours[j],
                                label='joint (stage 2)' if stage == 2 else pop)
            # Mock truth (shared values of stage 2) as a black cross; grey lines at zero.
            if kind == 'mock':
                tr = t[(t.stage == 2) & (t.population == 'shared')].set_index('param').truth
                ax.plot(tr[a], tr[b], 'kx', ms=9, mew=1.5, label='truth')
            ax.axhline(0, color='.7', lw=.5)
            ax.axvline(0, color='.7', lw=.5)
        # East to the left on the centre panel (reversed x axis), equal aspect.
        row[0].set(xlim=(1.4, -.6), ylim=(-1., 1.), aspect='equal', xlabel='centre offset east [deg]', ylabel='north [deg]',
                   title=f'{kind}, model: {model}')
        row[1].set(xlim=(-.25, .25), ylim=(-.25, .25), aspect='equal', xlabel=rf'${m[0][0]}_{{{m[0][1:]}}}$', ylabel=rf'${m[1][0]}_{{{m[1][1:]}}}$',
                   title='lopsided part' if m[0] == 'm1c' else 'm = 2 part')
    # One legend for all panels (labels collected, duplicates removed), to the right of the first row.
    handles = {}
    for ax in axes.flat:
        handles.update(dict(zip(*ax.get_legend_handles_labels()[::-1])))
    axes[0, 1].legend(handles.values(), handles.keys(), loc='upper left', bbox_to_anchor=(1.04, 1), fontsize=6.5, frameon=False)
    fig.savefig(ROOT / f'figures/fig_kin_mcmc_{kind}.png', dpi=200)
```

#### Run
Arguments: kind ('data' or 'mock'), model, and optionally 'dist' and 'outer' (see the header).

```python
main(sys.argv[1], sys.argv[2], 'dist' in sys.argv[3:], 'outer' in sys.argv[3:])
figure(sys.argv[1])
```

## What the models leave in the data

Figure 6.5 shows the residual velocities of the stars at 3° < ρ <
7° about three models, each with the rotation curve of every group fitted: the geometry and systemic motion of
vdMK14, the joint Level-1 fit and the joint Level-2 fit. In the frame of vdMK14 we refit each group's rotation curve, dispersions,
error scale and outlier fraction (maximum likelihood). About it the $m=1$ term dominates: 11–20
km/s in $v_t$, 7.2–12 km/s in $v_r$ and 7.6–9.9 km/s in
$v_{\rm los}$ (ranges over the giant groups; harmonic fits to the individual stars), in every group and in the RR Lyrae. Freeing the
centre, systemic motion and orientation (Level 1) reduces it to 1–7 km/s in all three
components: the $m=1$ pattern of the data is described by a displaced rotation centre and needs no lopsided rotation curve. What
remains is an $m=2$ pattern of 5.3–8.5 km/s in $v_t$ and
3.5–6.4 km/s in $v_r$, with the same phase in all groups (maximum of $v_t$ at position angle
145–164° in the five giant groups and 151° in the RR Lyrae). The streaming
terms of Level 2 absorb it, leaving 0.48–5.0 km/s in $v_t$ and
0.86–3.4 km/s in $v_r$ (the $m=1$ term of $v_t$ stays at
1.6–8.0 km/s); in Level 2 the rotation is slowest along the line of nodes ($m_{2c}$ =
-0.130).

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_kin_residual_profiles_ellipsoid.png')

**Figure 6.5.** Residual velocities (observed minus model, km/s) against position angle (deg) at
3° < ρ < 7°, median per 30° bin for each group (colours as in
Figure 6.4, legend above). Rows: tangential (a, d, g), radial (b, e, h) and line-of-sight (c, f, i)
components, with position angle and directions about each model's centre. Columns: the geometry and systemic motion of vdMK14 with
each group's fitted rotation curve (a–c); the joint Level-1 fit (d–f); the joint Level-2 fit with the velocity ellipsoid (g–i). In
each panel, the range over the giant groups of the $m=1$ and $m=2$ amplitudes $A_1$ and $A_2$ of a harmonic fit to the stars. Dotted
vertical lines: line of nodes; dash-dotted: minor axis. `scripts/paper_kin_residuals.py ellipsoid`

The bottom rows of Figure 6.1–Figure 6.3 show what Level 2 leaves on the sky, as
the significance $z$ of the residual in each cell. In the three metal-poor bins and the RR Lyrae the model is close to the data: the
cell values scatter with a standard deviation of 0.98–1.25 (1 for a perfect model), and at
most 3 per cent of the cells exceed $|z|=3$, still several times the 0.3 per cent expected for
unit normals (for the RR Lyrae, whose parameters are fitted beyond 4°, these cells lie mostly inside
4°). The two metal-rich bins, with the most stars and the smallest dispersions, are not: their cell values
scatter by 1.6–1.8, and 8–11 per cent of the
cells exceed $|z|=3$. The radial profile of the residuals (Figure 6.6) shows where: in the four
giant bins below [Fe/H] = -0.5 the stars at ρ = 3–5° rotate
2–5 km/s faster than the model and those at 2–3°
up to 6 km/s slower, so the rotation curve rises more steeply than the tanh form allows; beyond 7° the outer
slope $s$ removes the decline of 4–11 km/s that a pure tanh curve leaves in the three
metal-rich bins and the RR Lyrae (see the sky-aligned model in the checks), and beyond 5° $v_t$ agrees with the model to within
3.3 km/s for the giants. In $v_r$ the three bins above [Fe/H] = -1.1 move inwards by
2–4 km/s inside 4°, and the four bins above -1.5
outwards by up to 5–6 km/s beyond 6°, relative to the model, a radial flow that Level 2
does not contain. The RR Lyrae rotate more slowly than their model in 4 of the 5 rings at
4° < ρ < 9°, by 3–6 km/s, although their rotation
curve was fitted there; we have not traced the cause (the streaming they share with the giants in the joint frame, or the median of
residuals dominated by their large proper-motion errors), and it limits the precision of their $v_{\rm rot}$ to a few km/s.

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_kin_residual_radial_ellipsoid.png')

**Figure 6.6.** Median residual (observed minus the joint Level-2 model with the velocity
ellipsoid, km/s) of the tangential (a), radial (b) and line-of-sight (c) velocities in rings of angular distance ρ (deg) about the
fitted centre, one line per group (colours as in Figure 6.4, legend above; points of different groups
shifted slightly in ρ for clarity). The RR Lyrae have their parameters refitted at 4° < ρ <
9° in the joint frame, as in Figure 6.1. Bars: $\sqrt{\pi/2}$ × standard deviation /
$\sqrt n$, the error of the median of $n$ stars. `scripts/paper_kin_residuals.py ellipsoid`

The pattern is not an artefact of the fitting. In mocks drawn from symmetric discs about a known centre (positions from an exponential
disc in the footprint, Gaia errors of the nearest RR Lyrae, one realisation each, 3 for correlated errors;
`scripts/fig_kin_mock_physical.py`), with a thick disc, a hot disc with asymmetric drift, or spatially correlated Gaia errors, fits with
sky-aligned dispersions to proper motions and line-of-sight velocities put the centre within 0.10° of the true one
(proper motions with distances: up to 0.18° for the thick discs), and the fitted $m=2$ amplitude
$(m_{2c}^2+m_{2s}^2)^{1/2}$ stays at or below 0.05, against 0.09–0.16 in the
same fits to the data (0.10–0.18 with the velocity ellipsoid); its sine term lies between
-0.046 and +0.003, negative in all but 1 of the 8 fits.
The fitted centre does depend on which part of the disc a sample covers: in symmetric-disc fits (maximum likelihood, sky-aligned
dispersions, proper motions and line-of-sight velocities; `scripts/fig_kin_footprint.py`) of each giant bin, the east offsets span
0.55–1.09° on the bins' own sky positions and 0.65–1.09° when
every bin is resampled onto the positions of the most metal-poor bin; the two metal-rich bins move east by
0.17–0.18° (fit errors ≈ 0.09°), a mock symmetric disc by 0.07°.
Part of the spread of the stage-1 centres therefore comes from the different sky coverage of the bins sampling a non-axisymmetric
velocity field.

### Mocks drawn from symmetric discs

`scripts/fig_kin_mock_physical.py` makes the mocks of the previous paragraphs. Each mock places stars in three dimensions in a disc
with the vdMK14 centre and bulk motion and a known orientation: thin or thick, cold and isotropic or hot with asymmetric drift, or
cold with a spatially correlated field of proper-motion offsets added to the Gaia errors. Each star takes the Gaia proper-motion
covariance of the nearest RR Lyrae, and its velocity is projected on the sky at its own distance. The mocks are then fitted by maximum
likelihood with proper motions, line-of-sight velocities, distances and their combinations, with and without streaming. None of the
mocks has streaming, so any fitted $m=2$ term comes from the departure of the mock from the thin, cold model. The fitted parameters go
to `results/kin_mock_physical.csv`, from which the numbers above are written. The same drawing function makes the mocks of the
velocity-ellipsoid test in the checks.

<details><summary>About <code>scripts/fig_kin_mock_physical.py</code></summary>

```text
Can thickness, a hot anisotropic population with asymmetric drift, or spatially correlated Gaia proper-motion errors bias the
thin-disc fits (lmcold.kinfit) of the centre, inclination and line of nodes? Mocks that are NOT drawn from the fitting model.
Used in KB chapter 6 ('What the models leave in the data': the fitted centre and m = 2 amplitude of symmetric mock discs) and
chapter 7; the numbers are written from results/kin_mock_physical.csv by scripts/paper_text_kin_model.py and
scripts/paper_text_kin_support.py. Its draw() also makes the mocks of scripts/paper_kin_ellipsoid_mock.py.

Every mock has the adopted centre and bulk motion (lmcold.constants.LMC) and the disc orientation TRUTH (about the giants'
streaming fit, scripts/fig_kin_streaming.py). Positions are drawn in 3D: exponential surface density of scale length RD (R from a
Gamma(2, RD) distribution), uniform azimuth, height z ~ N(0, h); they are projected on the sky, and N_STARS stars at
2 < rho < 9 deg from the centre are kept (the RR Lyrae ring of scripts/fit_kinematics_mcmc.py). Drawing in 3D matters for the
distances: along one line of sight the stars above and below the plane lie at different R, so the radial gradient weights them
unequally (a fixed sky position with z of zero mean would hide that). Each star takes the Gaia proper-motion covariance of the
nearest RR Lyrae member (lmcold.samples.rrab). In the plane: radius R, unit vectors e_R and e_phi = n x e_R (the rotation sense
of kinfit). Velocities: v_R ~ N(0, sigma_R), v_phi ~ N(mean, sigma_phi), v_z ~ N(0, sigma_z), with mean = vc tanh(R/R0) for the cold
disc and, for the hot one, the asymmetric-drift (Jeans) mean
    mean^2 = vc^2 tanh^2(R/R0) - sigma_R^2 (R/RD - 1 + sigma_phi^2/sigma_R^2)
(exponential surface density of scale length RD, constant dispersions, no tilt term; floored at 0). The 3D velocity (systemic +
these) is projected on each star's sky basis at its own distance: proper motions with the Gaia covariance added as noise,
line-of-sight velocities (every star; error E_VLOS) and distance moduli 5 log10(D_star / D0) + N(0, E_MU).
Mocks (MOCKS): 'cold thin' (h 0, isotropic 20 km/s: the kinfit model up to the error model, a control); 'thick' (h H_THICK, cold
kinematics); 'hot' (h 0, sigma_R, sigma_phi, sigma_z = HOT and the drift); 'hot, no drift' (the same dispersions, mean
vc tanh(R/R0)); 'hot thick' (h H_THICK, HOT and the drift); 'correlated PM' (cold thin plus a
Gaussian random field of proper-motion offsets, rms PM_SYS per component, built from NMODES plane waves of wavelength WAVELENGTH deg,
NREAL realisations). Expectation for thickness: for a density stratified on similar oblate spheroids of axis ratio q the mean
distance along each line of sight lies on the plane conjugate to the line of sight, whose apparent tilt is below i (q = 0.5,
i = 28 deg: 20.4 deg); h = H_THICK at R = 2-8 kpc spans q ~ 0.1-0.5.

Fits (kinfit.fit, symmetric disc, i and theta free; centre fixed first, then free): 'PM' (kinfit.BASE), 'LOS' (centre, i, theta,
v0, R0, outliers, vlos0, sigma_los; bulk motion held at the 'PM' fit), 'PM + LOS', 'distances' (theta, i, dmu0, sigma_mu, outliers;
centre fixed) and 'PM + distances'; and 'PM + LOS stream', 'PM + distances stream': the same with the in-plane streaming
kinfit.STREAM free. None of the mocks has streaming, so any fitted m = 2 (|m2| = hypot(m2c, m2s); in the data 0.09-0.17 for the
giants, PM + LOS, with the v_phi minimum within 4 deg of the line of nodes in the metal-rich bins) is made by the mock's departure
from the thin, cold model (thickness, asymmetric drift, anisotropy, correlated errors).

Writes results/kin_mock_physical.csv (mock, realisation, data, nll, every parameter of kinfit.PARAMS and its error *_err) and
figures/fig_kin_mock_physical.png: four equal panels against the mock: (a) inclination, (b) line of nodes [deg], (c) centre
offset east and north [deg] (two markers); one marker per data set; truth as horizontal lines; (d) streaming amplitude |m2| of the
two streaming fits, with the range of the giants' data (grey band). Legend right of (d). Diagnostic figure (not in the paper).

Inputs: lmcold.samples (data/rrab.parquet, key release_rrab).

Run: .venv/bin/python scripts/fig_kin_mock_physical.py   (run time not recorded)
```
</details>

*Shown, not run here: the RR Lyrae catalogue of Zenodo bundle 1; maximum-likelihood fits of every mock and data set (run time not recorded). The cells below are the script itself; run it from the repository with* `python scripts/fig_kin_mock_physical.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The true disc orientation and rotation curve of the mocks, their dispersions and thickness, the error model, and the parameters
freed in each kind of fit.

```python
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from fit_kinematics_mcmc import ring
from lmcold import geometry as geo
from lmcold import kinfit, samples
from lmcold.constants import KAPPA, LMC
from lmcold.spheroid import spheroid
from lmcold.layout import panel_grid
from lmcold.sources import ROOT

TRUTH = dict(inc=28.0, theta=143.0)  # deg   # inclination, line of nodes (module docstring)
VC, R0, RD = 85.0, 2.0, 1.5  # km/s circular-speed amplitude, kpc turnover radius, kpc exponential scale length
COLD = (20.0, 20.0, 20.0)  # km/s sigma_R, sigma_phi, sigma_z
HOT = (40.0, 28.0, 30.0)  # km/s, sigma_phi / sigma_R ~ 1/sqrt(2) as for a flat rotation curve
H_THICK = 1.0  # kpc, Gaussian height dispersion
E_VLOS, E_MU = 2.0, 0.08  # km/s, mag
PM_SYS, WAVELENGTH, NMODES, NREAL = 0.025, 1.5, 200, 3  # mas/yr per component, deg, modes, realisations
MOCKS = {'cold thin': (0.0, COLD, False), 'thick': (H_THICK, COLD, False), 'hot': (0.0, HOT, True),
         'hot, no drift': (0.0, HOT, False), 'hot thick': (H_THICK, HOT, True), 'correlated PM': (0.0, COLD, False)}  # (h, dispersions, drift)
N_STARS = 15000  # stars per mock at 2 < rho < 9 deg
C = ('ra', 'dec', 'pmra', 'pmdec', 'pmra_error', 'pmdec_error', 'pmra_pmdec_corr')  # catalogue columns for kinfit.data
LOS, DIST = ('vlos0', 'sigma_los'), ('dmu0', 'sigma_mu')   # kinfit parameters added by velocities, distances
# Parameters freed in each kind of fit (names of kinfit.PARAMS); the rest stay at kinfit.start() (module docstring, 'Fits').
FREE = {'PM': kinfit.BASE, 'LOS': ('dx', 'dy', 'inc', 'theta', 'v0', 'R0', 'logit_fout') + LOS, 'PM + LOS': kinfit.BASE + LOS,
        'distances': ('inc', 'theta', 'logit_fout') + DIST, 'PM + distances': kinfit.BASE + DIST,
        'PM + LOS stream': kinfit.BASE + LOS + kinfit.STREAM, 'PM + distances stream': kinfit.BASE + DIST + kinfit.STREAM}
# Plot marker of each kind of fit.
MARKERS = {'PM': 's', 'LOS': '^', 'PM + LOS': 'o', 'distances': 'v', 'PM + distances': 'D', 'PM + LOS stream': 'P',
           'PM + distances stream': 'X'}
M2_DATA = (0.094, 0.171)  # |m2| range of the giants' PM + LOS streaming fits (results/kin_tilted_ring.csv, model 'stream')
```

#### Drawing a mock
`draw` places N_STARS stars in three dimensions (an exponential disc, optionally thick, or a spheroid), gives each the Gaia errors
of the nearest RR Lyrae, draws velocities about the mean rotation (with or without asymmetric drift) and projects them on the sky.

```python
def draw(c, h, disp, drift, rng, pm_field=False, q=None):
    """Mock catalogue dict (C columns plus vlos, evlos, dmu): positions drawn in 3D, Gaia errors of the nearest star of c (see the
    module docstring); with q, positions from an oblate K0 spheroid of axis ratio q instead (lmcold.spheroid.spheroid, face-on
    exponential of scale length RD, its thickness growing with R as for the old population of paper Section 5; h is then unused).

    Parameters
    ----------
    c : dict
        Catalogue whose positions (ra, dec [deg]) and proper-motion errors (pmra_error, pmdec_error [mas/yr], pmra_pmdec_corr) give
        each mock star its errors (nearest star on the sky).
    h : float
        Gaussian height dispersion [kpc]; 0 for a thin disc.
    disp : tuple of float
        (sigma_R, sigma_phi, sigma_z) [km/s], constant over the disc.
    drift : bool
        Mean rotation lagging vc tanh(R/R0) by the asymmetric drift (module docstring) if True, vc tanh(R/R0) if False.
    rng : numpy.random.Generator
    pm_field : bool
        Add the correlated proper-motion field (PM_SYS, WAVELENGTH, NMODES).
    q : float or None
        Axis ratio of a K0 spheroid for the positions instead of the disc.

    Returns
    -------
    dict
        Arrays of length N_STARS: ra, dec [deg], pmra, pmdec [mas/yr], their errors and correlation, vlos, evlos [km/s], dmu [mag].
    """
    # Unit vectors (3,): line of sight to the centre, east and north there, line of nodes, and the disc normal (geo.symmetry_axis).
    r0, eE0, eN0, line, n = (np.asarray(v) for v in geo.symmetry_axis(np.deg2rad(LMC['ra0']), np.deg2rad(LMC['dec0']),
                                                                       np.deg2rad(TRUTH['inc']), np.deg2rad(TRUTH['theta'])))
    # LMC distance [kpc] (lmcold.constants).
    D0 = LMC['D0']
    m = 20 * N_STARS  # draw many, keep N_STARS in the ring
    if q is None:
        # In-plane radius [kpc] from Gamma(2, RD) (exponential surface density), azimuth [rad] uniform, height z [kpc] ~ N(0, h).
        Rd, ph = rng.gamma(2.0, RD, m), rng.uniform(0, 2 * np.pi, m)
        z = rng.normal(0, h, m) if h > 0 else np.zeros(m)
    else:
        # Spheroid positions (m, 3) [kpc] in the disc frame, converted to the same R, phi, z.
        X = spheroid(RD, q, m, rng)
        Rd, ph, z = np.hypot(X[:, 0], X[:, 1]), np.arctan2(X[:, 1], X[:, 0]), X[:, 2]
    # Position relative to the LMC centre [kpc], (m, 3): in-plane axes line and n x line, plus z along the normal.
    pos = Rd[:, None] * (np.cos(ph)[:, None] * line + np.sin(ph)[:, None] * np.cross(n, line)) + z[:, None] * n
    xyz = D0 * r0 + pos  # heliocentric position [kpc]
    Ds = np.linalg.norm(xyz, axis=1)
    rhat = xyz / Ds[:, None]
    # Angular distance from the centre [deg]; the first N_STARS stars at 2 < rho < 9 deg are kept (the ring of the module
    # docstring).
    rho = np.rad2deg(np.arccos(np.clip(rhat @ r0, -1, 1)))
    keep = np.flatnonzero((rho > 2) & (rho < 9))[:N_STARS]
    rhat, Ds, pos = rhat[keep], Ds[keep], pos[keep]
    # Sky coordinates [deg] of each mock star; then the nearest catalogue star on the unit sphere gives its proper-motion errors.
    ra, dec = np.rad2deg(np.arctan2(rhat[:, 1], rhat[:, 0])) % 360, np.rad2deg(np.arcsin(rhat[:, 2]))
    _, near = cKDTree(np.c_[np.cos(np.deg2rad(c['dec'])) * np.cos(np.deg2rad(c['ra'])),
                            np.cos(np.deg2rad(c['dec'])) * np.sin(np.deg2rad(c['ra'])), np.sin(np.deg2rad(c['dec']))]).query(rhat)
    c = {k: c[k][near] for k in ('pmra_error', 'pmdec_error', 'pmra_pmdec_corr')} | dict(ra=ra, dec=dec)
    # kinfit.data with zero proper motions, used only for its sky basis (ea, ed: east and north unit vectors, (N, 3)) and the
    # proper-motion covariance matrices cov, (N, 2, 2) [(mas/yr)^2].
    d = kinfit.data(ra, dec, np.zeros(len(ra)), np.zeros(len(ra)), c['pmra_error'], c['pmdec_error'], c['pmra_pmdec_corr'])
    ea, ed = np.asarray(d['ea']), np.asarray(d['ed'])
    D = Ds
    # In-plane radius R [kpc] and unit vectors e_R, e_phi = n x e_R (the rotation sense of kinfit).
    inplane = pos - (pos @ n)[:, None] * n
    R = np.linalg.norm(inplane, axis=1)
    eR = inplane / R[:, None]
    ephi = np.cross(n, eR)
    sR, sphi, sz = disp
    # Mean v_phi [km/s]: vc tanh(R/R0), or with drift the asymmetric-drift mean of the module docstring (floored at 0).
    vc = VC * np.tanh(R / R0)
    mean = np.sqrt(np.maximum(vc**2 - sR**2 * (R / RD - 1 + sphi**2 / sR**2), 0)) if drift else vc
    # Velocities [km/s] in the plane; V (N, 3): the systemic velocity (line-of-sight vlos0 plus the bulk proper motion at D0) plus
    # these.
    vR, vphi, vz = (rng.normal(m, s, len(D)) for m, s in ((0, sR), (mean, sphi), (0, sz)))
    V = LMC['vlos0'] * r0 + KAPPA * D0 * (LMC['pmra0'] * eE0 + LMC['pmdec0'] * eN0) + vR[:, None] * eR + vphi[:, None] * ephi \
        + vz[:, None] * n
    # Proper motions [mas/yr] at each star's own distance Ds [kpc] (mu = v_t / (KAPPA D)), plus Gaia noise drawn from its
    # covariance.
    pmra, pmdec = (V * ea).sum(1) / (KAPPA * Ds), (V * ed).sum(1) / (KAPPA * Ds)
    noise = np.einsum('nij,nj->ni', np.linalg.cholesky(np.asarray(d['cov'])), rng.normal(size=(len(D), 2)))
    pmra, pmdec = pmra + noise[:, 0], pmdec + noise[:, 1]
    if pm_field:
        # Gaussian random field: NMODES plane waves of wavelength WAVELENGTH deg in random directions and phases on the tangent
        # plane (x east, y north [deg]), scaled to rms PM_SYS per component.
        x, y = (rhat - r0) @ eE0 * 180 / np.pi, (rhat - r0) @ eN0 * 180 / np.pi
        for comp in range(2):
            ang, ph = rng.uniform(0, 2 * np.pi, NMODES), rng.uniform(0, 2 * np.pi, NMODES)
            k = 2 * np.pi / WAVELENGTH
            f = np.sqrt(2.0 / NMODES) * np.cos(k * (np.outer(x, np.cos(ang)) + np.outer(y, np.sin(ang))) + ph).sum(1)
            if comp == 0:
                pmra = pmra + PM_SYS * f
            else:
                pmdec = pmdec + PM_SYS * f
    # Line-of-sight velocity [km/s] with error E_VLOS; distance modulus relative to D0 [mag] with error E_MU.
    vlos = (V * rhat).sum(1) + rng.normal(0, E_VLOS, len(D))
    dmu = 5 * np.log10(Ds / D0) + rng.normal(0, E_MU, len(D))
    return dict(c, pmra=pmra, pmdec=pmdec, vlos=vlos, evlos=np.full(len(D), E_VLOS), dmu=dmu)
```

#### Fitting a mock
`fit` fits one data set of a mock (proper motions, line-of-sight velocities, distances, or combinations) by maximum likelihood.

```python
def fit(m, kind, bulk=None):
    """Fit data set `kind` (a key of FREE) of mock m, the data not used set to NaN; bulk (pmra0, pmdec0 [mas/yr]) holds the bulk
    motion for 'LOS'. Returns (p, err, nll): best-fit vector and its inverse-Hessian errors (shape (33,) each, kinfit.PARAMS order)
    and the negative log likelihood."""
    # The data not used in this kind of fit are set to NaN (pmra NaN: no proper motions for that star).
    nan = np.full(len(m['ra']), np.nan)
    pmra = nan if kind in ('LOS', 'distances') else m['pmra']
    vlos = m['vlos'] if 'LOS' in kind else nan
    dmu = m['dmu'] if 'distances' in kind else nan
    # Arrays for the likelihood (positions [deg], proper motions [mas/yr], v_los [km/s], distance moduli [mag]).
    d = kinfit.data(m['ra'], m['dec'], pmra, m['pmdec'], m['pmra_error'], m['pmdec_error'], m['pmra_pmdec_corr'], vlos,
                    m['evlos'], dmu)
    # Start vector (33,); for 'LOS', the bulk motion pmra0, pmdec0 (indices 2, 3) held at the 'PM' fit.
    p0 = kinfit.start()
    if bulk is not None:
        p0[2:4] = bulk
    free = FREE[kind]
    # When the centre (dx, dy) is free, first fit with it fixed, then start the full fit from there.
    if free[:2] == ('dx', 'dy'):
        p0, _, _ = kinfit.fit(d, free=free[2:], p0=p0)
    return kinfit.fit(d, free=free, p0=p0)
```

#### Load the data, draw and fit every mock, and draw the figure
`main` reads the RR Lyrae members (`lmcold.samples.rrab()`, Zenodo bundle 1) for the footprint and errors, draws each mock
(three realisations of the correlated field), fits every data set, and writes `results/kin_mock_physical.csv` and the figure.

```python
def main():
    """Draw every mock, fit every data set, write the csv and the figure of the module docstring."""
    rr = samples.rrab()
    # The RR Lyrae at 2 < rho < 9 deg: footprint and Gaia errors of the mocks.
    c = ring({k: rr[k] for k in C})
    rows = []
    for name, (h, disp, drift) in MOCKS.items():
        for real in range(NREAL if name == 'correlated PM' else 1):
            # One mock per realisation (seed 100 + realisation); then every kind of fit, the 'PM' fit first so its bulk motion feeds
            # 'LOS'.
            m = draw(c, h, disp, drift, np.random.default_rng(100 + real), pm_field=name == 'correlated PM')
            bulk = None
            for kind in FREE:
                p, e, nll = fit(m, kind, bulk if kind == 'LOS' else None)
                if kind == 'PM':
                    bulk = p[2:4]
                rows.append(dict(mock=name, realisation=real, data=kind, nll=nll, **dict(zip(kinfit.PARAMS, p)),
                                 **{f'{n}_err': v for n, v in zip(kinfit.PARAMS, e)}))
                # Progress line; the indices follow kinfit.PARAMS (0 dx, 1 dy, 4 inc, 5 theta, 6 v0, 8 sigma, 12 sigma_los, 17-21
                # streaming).
                print(name, real, kind, f'i {p[4]:.1f}+-{e[4]:.1f} theta {p[5]:.1f} dx {p[0]:.3f}+-{e[0]:.3f} dy {p[1]:.3f}+-{e[1]:.3f}'
                      f' v0 {p[6]:.1f} sigma {p[8]:.1f} sigma_los {p[12]:.1f} nll {nll:.1f} m2c {p[17]:.3f} m2s {p[18]:.3f}'
                      f' vr0 {p[19]:.3f} r2c {p[20]:.3f} r2s {p[21]:.3f}', flush=True)
    t = pd.DataFrame(rows)
    t.to_csv(ROOT / 'results/kin_mock_physical.csv', index=False, float_format='%.5f')

    plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
    # One x position per mock (and per realisation of the correlated field); four equal panels in a row.
    labels = [f'{r.mock}' + (f' {r.realisation}' if r.mock == 'correlated PM' else '') for _, r in
              t.drop_duplicates(['mock', 'realisation']).iterrows()]
    fig, ax = panel_grid(1, 4, width=9.0, left=0.5, right=1.3, hgap=0.55, top=0.15, bottom=0.75)
    for k, ((name, real), s) in enumerate(t.groupby(['mock', 'realisation'], sort=False)):
        for j, kind in enumerate(FREE):
            r = s[s.data == kind].iloc[0]
            # The fits of one mock spread around its x position; (a) inclination, (b) line of nodes, (d) |m2| of the streaming fits,
            # (c) centre offsets east (red) and north (blue, open) of every fit with a centre.
            x = k + 0.12 * (j - 2)
            ax[0, 0].errorbar(x, r.inc, yerr=r.inc_err, fmt=MARKERS[kind], ms=3, lw=0.6, color='k',
                              label=kind if k == 0 else None)
            ax[0, 1].errorbar(x, r.theta, yerr=r.theta_err, fmt=MARKERS[kind], ms=3, lw=0.6, color='k')
            if 'stream' in kind:
                ax[0, 3].plot(x, np.hypot(r.m2c, r.m2s), MARKERS[kind], ms=3, color='k')
            if kind != 'distances':
                ax[0, 2].errorbar(x, r.dx, yerr=r.dx_err, fmt=MARKERS[kind], ms=3, lw=0.6, color='tab:red')
                ax[0, 2].errorbar(x, r.dy, yerr=r.dy_err, fmt=MARKERS[kind], ms=3, lw=0.6, color='tab:blue', mfc='white')
    # Truth as horizontal grey lines, mock names as tick labels.
    for a, v, lab in ((ax[0, 0], TRUTH['inc'], r'inclination $i$ [deg]'), (ax[0, 1], TRUTH['theta'], r'line of nodes $\theta$ [deg]'),
                      (ax[0, 2], 0.0, 'centre offset [deg]')):
        a.axhline(v, color='0.5', lw=0.6)
        a.set(ylabel=lab, xticks=range(len(labels)), xlim=(-0.5, len(labels) - 0.5))
        a.set_xticklabels(labels, rotation=60, ha='right', fontsize=5.5)
    # Panel (d): the giants' |m2| range as a grey band.
    ax[0, 3].axhspan(*M2_DATA, color='0.85', lw=0)
    ax[0, 3].set(ylabel='|m2| [fraction of vrot]', xticks=range(len(labels)), xlim=(-0.5, len(labels) - 0.5), ylim=(0, 0.2))
    ax[0, 3].set_xticklabels(labels, rotation=60, ha='right', fontsize=5.5)
    # Legend entries for the band and the two centre markers; one legend right of panel (d), outside the data.
    ax[0, 3].plot([], [], 's', color='0.85', ms=5, label='giants data, |m2|')
    ax[0, 2].plot([], [], 'o', color='tab:red', ms=3, label='east (red)')
    ax[0, 2].plot([], [], 'o', color='tab:blue', mfc='white', ms=3, label='north (blue, open)')
    h0, l0 = ax[0, 0].get_legend_handles_labels()
    h2, l2 = ax[0, 2].get_legend_handles_labels()
    h3, l3 = ax[0, 3].get_legend_handles_labels()
    ax[0, 3].legend(h0 + h2 + h3, l0 + l2 + l3, loc='upper left', bbox_to_anchor=(1.03, 1), fontsize=5.5, frameon=False)
    fig.savefig(ROOT / 'figures/fig_kin_mock_physical.png', dpi=200)
```

#### Run

```python
main()
```

### Density centres

The kinematic centre is compared in chapter 7 with the centre of the star counts. `scripts/fig_density_centre.py` measures that
density centre for the RR Lyrae, for the *Gaia* red giants that the BOSS carton selects, and for the BOSS giants of each [Fe/H] bin
weighted for selection, coverage and Milky Way foreground: the weighted mean position of the stars within an aperture, iterated from
the vdMK14 centre until it stops moving, for several aperture radii, with errors from bootstrap resamples of the stars. It writes
`results/density_centre.csv`, read by the centre figure and the text of chapter 7.

<details><summary>About <code>scripts/fig_density_centre.py</code></summary>

```text
Density centres of the RR Lyrae and of the giants by [Fe/H], to compare with the kinematic centre.

Centre: the weighted mean position of the stars within APERTURES[j] deg of the current centre, iterated from the adopted centre
until it moves less than 1e-4 deg; the centre of point symmetry of the counts within that radius (an inclined disc projects to
ellipses, symmetric about its centre). Positions on the tangent plane of the adopted centre (lmcold.footprint.lmc_xy).
Samples: RRab members (complete within 10 deg; lmcold.samples); the photometric red giants that the BOSS carton selects from
Gaia (data/boss_parent.parquet, `eligible`: G <= 17.5, colour, magnitude and proper-motion cuts, all [Fe/H]); and the BOSS giants
of lmcold.mass_samples weighted for selection, coverage and Milky Way foreground (w x fill x (1 - f_mw)), in the [Fe/H] bins of
lmcold.samples.FEH_BINS.
Errors: 16th-84th percentiles of NBOOT bootstrap resamples. Writes results/density_centre.csv and figures/fig_density_centre.png
(offset east and north against aperture, with the kinematic centres of results/kin_mcmc_data_both.csv, stage 1, in the margin).

Diagnostic for the kinematic centre (Sect. 3); not a paper figure. results/density_centre.csv is read by scripts/paper_kin_centre.py
(the density centres of figure paper_kin_centre, KB chapter 7) and scripts/paper_text_kin_support.py (distances of the kinematic
centres from the density centres).

Inputs: data/rrab.parquet (sources.yaml key release_rrab, via lmcold.samples.rrab); data/boss_parent.parquet
(release_boss_parent); the weighted BOSS giants of lmcold.mass_samples.samples (release_boss_weights and the files listed there);
data/giant_foreground.parquet (coverage fill and Milky Way foreground fraction f_mw per BOSS giant, scripts/build_giant_mass.py);
results/kin_mcmc_data_both.csv (scripts/fit_kinematics_mcmc.py; columns stage, population, param, median, p16, p84).

Figure (two equal square panels; legend right of (b)): x: aperture radius [deg]; each population is shifted by 0.06 deg in x for
legibility. y: density-centre offset east (left panel) and north (right panel) of the adopted centre [deg] on the orthographic
tangent plane, with 16th-84th bootstrap percentiles. Stars right of the dotted line at x = 5.6: stage-1 kinematic centre (median
and 16th-84th percentiles of dx or dy) of the same population, where results/kin_mcmc_data_both.csv has one.

results/density_centre.csv: population, aperture [deg], N (stars within the aperture of the final centre), dx, dy [deg] and their
16th and 84th bootstrap percentiles (dx_p16, dx_p84, dy_p16, dy_p84).

Run: .venv/bin/python scripts/fig_density_centre.py   (run time not recorded)
```
</details>

*Shown, not run here: the RR Lyrae, BOSS parent and weighted BOSS giant catalogues of Zenodo bundle 1; bootstrap resampling (run time not recorded). The cells below are the script itself; run it from the repository with* `python scripts/fig_density_centre.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The aperture radii and the number of bootstrap resamples.

```python
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lmcold import samples
from lmcold.footprint import lmc_xy
from lmcold.layout import panel_grid
from lmcold.mass_samples import samples as mass_samples
from lmcold.sources import DATA, ROOT

APERTURES = [1.5, 2.5, 3.5, 5.]  # deg
NBOOT = 200  # bootstrap resamplings of the stars
```

#### The density centre within one aperture
`centre` iterates the weighted mean position of the stars within radius `r` of the current centre until it stops moving.

```python
def centre(x, y, w, r):
    """Iterated weighted mean position within radius r, starting from the origin (the adopted centre).

    Parameters
    ----------
    x, y : numpy.ndarray
        Tangent-plane coordinates east and north of the adopted centre [deg].
    w : numpy.ndarray
        Weights per star.
    r : float
        Aperture radius [deg].

    Returns
    -------
    numpy.ndarray, shape (2,)
        (x, y) of the centre [deg]: the first iterate that moves less than 1e-4 deg, or the 200th.
    """
    c = np.zeros(2)                    # start at the adopted centre [deg]
    for _ in range(200):               # at most 200 iterations
        # Stars within r of the current centre and their weighted mean position [deg].
        k = np.hypot(x - c[0], y - c[1]) < r
        new = np.array([np.average(x[k], weights=w[k]), np.average(y[k], weights=w[k])])
        # Converged when the centre moves by less than 1e-4 deg.
        if np.hypot(*(new - c)) < 1e-4:
            break
        c = new
    return new
```

#### Load the data
The RR Lyrae members (`lmcold.samples.rrab()`, `data/rrab.parquet`), the Gaia red giants the BOSS carton selects
(`data/boss_parent.parquet`), and the BOSS giants with their selection weights (`lmcold.mass_samples`) joined to their coverage
fill and Milky Way foreground fraction (`data/giant_foreground.parquet`); all from Zenodo bundle 1. The giants are split into the
five [Fe/H] bins of the kinematic fits.

```python
rr = samples.rrab()   # dict of RRab member arrays (ra, dec [deg], ...)
# Gaia stars the BOSS carton selects (lmcold.magcloud.eligible), unweighted, all metallicities.
par = pd.read_parquet(DATA / 'boss_parent.parquet').query('eligible')
# Populations: (label, ra [deg], dec [deg], weight per star); RR Lyrae and Gaia giants unweighted.
pops = [('RR Lyrae', rr['ra'], rr['dec'], np.ones(len(rr['ra']))),
        ('red giants, Gaia G<17.5', par.ra.to_numpy(), par.dec.to_numpy(), np.ones(len(par)))]
# BOSS giants with selection weights w (first element of lmcold.mass_samples.samples), joined to fill and f_mw per star.
b = mass_samples()[0].merge(pd.read_parquet(DATA / 'giant_foreground.parquet')[['source_id', 'fill', 'f_mw']], on='source_id',
                            validate='one_to_one')
# Weight: selection weight x coverage fill x (1 - Milky Way foreground fraction).
b = b.assign(W=b.w * b['fill'] * (1 - b.f_mw))
# One population per [Fe/H] bin of lmcold.samples.FEH_BINS; the open end bins have edges -9 and 9, labelled as one-sided.
for lo, hi in samples.FEH_BINS:
    s = b[(b.feh >= lo) & (b.feh < hi)]
    label = f"[Fe/H] < {hi}" if lo < -5 else (f"[Fe/H] > {lo}" if hi > 5 else f"{lo} < [Fe/H] < {hi}")
    pops.append((f'giants {label}', s.ra.to_numpy(), s.dec.to_numpy(), s.W.to_numpy()))
```

#### Density centres and their bootstrap errors
For each population and aperture: the centre, and the 16th-84th percentiles of the centres of NBOOT bootstrap resamples.

```python
rows = []                          # one dict per (population, aperture)
rng = np.random.default_rng(1)     # fixed seed of the bootstrap
for name, ra, dec, w in pops:
    x, y = lmc_xy(ra, dec)   # deg east and north of the adopted centre, tangent plane
    for r in APERTURES:
        c = centre(x, y, w, r)
        # Bootstrap: resample stars with replacement and repeat the iteration.
        boot = np.array([centre(x[i], y[i], w[i], r) for i in (rng.integers(0, len(x), len(x)) for _ in range(NBOOT))])
        lo, hi = np.percentile(boot, [16, 84], axis=0)
        rows.append(dict(population=name, aperture=r, N=int((np.hypot(x - c[0], y - c[1]) < r).sum()), dx=c[0], dy=c[1],
                         dx_p16=lo[0], dx_p84=hi[0], dy_p16=lo[1], dy_p84=hi[1]))
        print(rows[-1], flush=True)
# Table of centres, written to results/density_centre.csv.
t = pd.DataFrame(rows)
t.to_csv(ROOT / 'results/density_centre.csv', index=False, float_format='%.4f')
```

#### The figure
Density-centre offsets east and north against aperture radius, with the stage-1 kinematic centres of
`results/kin_mcmc_data_both.csv` (from `scripts/fit_kinematics_mcmc.py data both`) in the margin.

```python
kin = pd.read_csv(ROOT / 'results/kin_mcmc_data_both.csv').query('stage == 1')   # kinematic fits: centre offsets dx, dy [deg]
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
fig, axes = panel_grid(1, 2, width=7, left=.6, right=1.9, hgap=.6, bottom=.5, top=.3, aspect=1.)
# One colour per population, in the order of pops: RR Lyrae, Gaia red giants, five giant [Fe/H] bins.
colours = ['#7b3294', 'k', '#d7191c', '#fdae61', '#a6d96a', '#1a9641', '#2878b5']
for ax, c in zip(axes[0], ('dx', 'dy')):
    for j, (name, colour) in enumerate(zip(t.population.unique(), colours)):
        s = t[t.population == name]
        # Aperture shifted by 0.06 deg per population for legibility; error bars are the bootstrap 16th-84th percentiles.
        a = s.aperture + .06 * (j - 3)
        ax.errorbar(a, s[c], [s[c] - s[f'{c}_p16'], s[f'{c}_p84'] - s[c]], fmt='o-', ms=4, lw=1, color=colour, label=name)
        # Stage-1 kinematic centre of the same population (if fitted), drawn as a star at x = 5.9 + 0.06 j.
        k = kin[(kin.population == name) & (kin.param == c)]
        if len(k):
            k = k.iloc[0]
            ax.errorbar(5.9 + .06 * j, k['median'], [[k['median'] - k.p16], [k.p84 - k['median']]], fmt='*', ms=8, color=colour)
    # Zero offset (adopted centre) and the dotted divider between apertures and kinematic centres at x = 5.6.
    ax.axhline(0, color='.5', lw=.6)
    ax.axvline(5.6, color='.7', lw=.6, ls=':')
    ax.set(xlim=(1, 6.4), xlabel='aperture radius [deg]  (stars: kinematic)', ylabel=f"density centre offset {'east' if c == 'dx' else 'north'} [deg]")
axes[0, 1].legend(loc='upper left', bbox_to_anchor=(1.03, 1), frameon=False, fontsize=7)
fig.savefig(ROOT / 'figures/fig_density_centre.png', dpi=200)
```

## Checks

### Recovery of the velocity ellipsoid from mocks

We drew mocks of 15 000 stars at 2° < ρ < 9° from five physical discs: positions in
three dimensions from an exponential disc of scale length $h$ = 1.5 kpc (thin, or with a Gaussian height dispersion of
1 kpc), or from a spheroid with density $\propto K_0(m/h)$ ($K_0$ the modified Bessel function of the second kind,
$m^2=R^2+z^2/q^2$ the spheroidal radius, $z$ the height above the plane; its face-on profile is exponential with the same $h$) of axis
ratio $q$ = 0.5 ('hot spheroid', the shape of the old population found in chapter 8; rms height 1.3 kpc, growing
with radius), Gaia covariances of the nearest RR Lyrae member, line-of-sight errors of 2 km/s, and constant
dispersions $(\sigma_R,\sigma_\phi,\sigma_z)$ = (20, 20, 20) km/s ('cold
thin') or (40, 28, 30) km/s (the hot thin and hot thick discs and the hot
spheroid). The mean rotation is $v_c\tanh(R/R_0)$ (with $v_c$ the amplitude of the mock's circular-speed curve) for the cold disc and
one hot disc ('hot, no drift'), and $v_c\tanh(R/R_0)$ minus the asymmetric drift for the other three, so theirs is not of the fitted
form. None of the mocks has streaming or a dispersion gradient; the fits have both free. Figure 6.7 shows the
maximum-likelihood dispersions. With proper motions and line-of-sight velocities, $\sigma_R$ is recovered to within
-0.9 to +0.7 km/s, $\sigma_\phi$ to within 1.3 km/s (biased high) and $\sigma_z$ to
within 0.5 km/s, with statistical errors of 0.2–0.7 km/s; with a pure tanh curve
($s=0$) the dispersions change by less than 0.5 km/s. With proper motions alone, $\sigma_R$ and $\sigma_\phi$ are
recovered to within 2.5 km/s (thick disc: $\sigma_\phi$ 4.8 km/s too high), and $\sigma_z$
comes out 27–36 km/s for true values of 20–30 km/s, with
errors of 4.2–5.5 km/s: the proper motions do not measure $\sigma_z$. The hot spheroid is recovered
as well as the discs with line-of-sight velocities (dispersions within 0.6 km/s, inclination 27.7°
for 28°, line of nodes 140° for 143°); with proper motions alone its
$\sigma_z$ comes out at 50 ± 8 km/s and its inclination at 17°.

#### Drawing and fitting the mocks

The mocks are drawn and fitted by `scripts/paper_kin_ellipsoid_mock.py`, which also makes Figure 6.7 (about 20
minutes on one machine, so it is not run on this page: `python scripts/paper_kin_ellipsoid_mock.py`). It draws each mock with the
drawing function of `scripts/fig_kin_mock_physical.py` and fits it three times by maximum likelihood with the model of this page:
with proper motions and line-of-sight velocities, with the outer slope $s$ held at zero, and with proper motions alone.

<details><summary>About <code>scripts/paper_kin_ellipsoid_mock.py</code></summary>

```text
Does the disc-frame velocity-ellipsoid likelihood (lmcold.kinfit, ellipsoid=True) recover sigma_R, sigma_phi and sigma_z from mocks
that are not drawn from it? Acceptance test of the ellipsoid model of the kinematics section (appendix; KB chapter 6, 'Recovery of
the velocity ellipsoid from mocks', figure paper_kin_ellipsoid_mock; the recovery numbers there are written from
results/kin_ellipsoid_mock.csv by scripts/paper_text_kin_model.py).

Mocks: the physical mocks of scripts/fig_kin_mock_physical.py (draw(); positions drawn in 3D from an exponential disc, Gaia
covariances of the nearest RR Lyrae member, line-of-sight errors 2 km/s, 15 000 stars at 2 < rho < 9 deg), with constant
dispersions: 'cold thin' (20, 20, 20 km/s, thin, mean vc tanh(R/R0): the fitting model up to its rotation-curve shape), 'hot'
(40, 28, 30, thin, asymmetric-drift mean), 'hot, no drift' (40, 28, 30, mean vc tanh(R/R0)) and 'hot thick' (40, 28, 30, Gaussian
height dispersion 1 kpc, drift) and 'hot spheroid' (40, 28, 30, drift, positions from an oblate K0 spheroid of axis ratio Q_SPH:
the rotation-implied shape of the metal-poor giants, paper Section 5, whose thickness grows with R). None has streaming or a radial dispersion gradient; the fit has both free.
Fits: maximum likelihood of the Level-2 ellipsoid model with everything free as for the data (fit_kinematics_mcmc.ml, model
'ellipsoid', outer slope vslope of the rotation curve free): 'PM + LOS' (the giants' case) and 'PM' (proper motions only, the
RR Lyrae case; sigma_z then comes only from the proper motions); and 'PM + LOS, tanh': the same with vslope held at 0, a pure tanh
rotation curve, to show what the slope removes. Errors from the inverse Hessian.

Writes results/kin_ellipsoid_mock.csv (mock, data, truth and fitted sigma_R, sigma_phi, sigma_z with errors, dlnsig_dR, vslope, inc,
theta, v0) and figures/paper_kin_ellipsoid_mock.png: three equal panels (sigma_R, sigma_phi, sigma_z [km/s]) against the mock;
filled circles PM + LOS, open circles PM only, grey squares PM + LOS with a tanh curve, truth as red bars. Paper figure (appendix app:kin).

Inputs: the RR Lyrae members of lmcold.samples.rrab() (data/rrab.parquet, Zenodo bundle 1): their sky positions set the footprint
and their Gaia proper-motion covariances the mock errors. Run time: about 20 min on one machine (docs/release_scripts.md).

Run: .venv/bin/python scripts/paper_kin_ellipsoid_mock.py
```
</details>

**Data read by `scripts/paper_kin_ellipsoid_mock.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/rrab.parquet` | Zenodo bundle 1 | RRab within 20 deg: OGLE-IV + Gaia DR3, [Fe/H], dmu, membership |

*Shown, not run here: the RR Lyrae catalogue of Zenodo bundle 1; about 20 min. The cells below are the script itself; run it from the repository with* `python scripts/paper_kin_ellipsoid_mock.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

##### Settings
The names of the five mocks (their dispersions and shapes are set in `fig_kin_mock_physical.MOCKS`), the axis ratio of the
spheroid, and the three dispersions compared.

```python
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fig_kin_mock_physical import C, HOT, MOCKS, draw
from fit_kinematics_mcmc import ml, ring
from lmcold import kinfit, samples
from lmcold.layout import panel_grid
from lmcold.sources import ROOT

NAMES = ('cold thin', 'hot', 'hot, no drift', 'hot thick', 'hot spheroid')   # the five mocks, in plotting order (module docstring)
Q_SPH = 0.5  # axis ratio of the 'hot spheroid' mock
COMPS = ('sigma_R', 'sigma_phi', 'sigma_z')  # km/s: the disc-frame dispersions compared with the truth
LABELS = dict(sigma_R=r'$\sigma_R$', sigma_phi=r'$\sigma_\phi$', sigma_z=r'$\sigma_z$')   # y-axis labels of the panels
```

##### Load the data
The RR Lyrae members (`lmcold.samples.rrab()`, from `data/rrab.parquet`) at 2-9 deg: only their positions and Gaia proper-motion
errors are used, as the footprint and error model of the mocks.

```python
# RR Lyrae members (dict of columns): only positions [deg] and proper-motion errors [mas/yr] are used
rr = samples.rrab()
c = ring({k: rr[k] for k in C})   # positions and proper-motion errors (columns C) at 2 < rho < 9 deg
```

##### Draw the mocks and fit them
Each mock takes the sky positions and Gaia errors of the RR Lyrae members (`fig_kin_mock_physical.draw`); the 'hot spheroid'
draws its positions from a $K_0$ spheroid of axis ratio `Q_SPH`. Each is fitted three times by maximum likelihood: proper motions
and line-of-sight velocities, proper motions only, and both with a pure tanh rotation curve. This step takes a while.

```python
# One output row per (mock, kind of fit)
rows = []
for name in NAMES:
    # Spheroid: HOT dispersions with asymmetric drift, positions from the K0 spheroid of axis ratio Q_SPH;
    # the other mocks: (height dispersion h [kpc], dispersions [km/s], drift flag) from MOCKS; same seed 100 for all
    if name == 'hot spheroid':
        disp = HOT
        m = draw(c, 0.0, HOT, True, np.random.default_rng(100), q=Q_SPH)
    else:
        h, disp, drift = MOCKS[name]
        m = draw(c, h, disp, drift, np.random.default_rng(100))
    for data in ('PM + LOS', 'PM', 'PM + LOS, tanh'):
        # Line-of-sight velocities [km/s] kept for 'PM + LOS' fits, NaN (missing) for the proper-motion-only fit
        vlos = m['vlos'] if data.startswith('PM + LOS') else np.full(len(m['ra']), np.nan)
        d = kinfit.data(*(m[k] for k in C), vlos, m['evlos'])
        # Maximum-likelihood fit of the ellipsoid model; p: kinfit.PARAMS vector
        p = ml(d, 'ellipsoid')
        if data.endswith('tanh'):  # refit with a pure tanh rotation curve, everything else free as before
            p[kinfit.PARAMS.index('vslope')] = 0.0
            free_all = tuple(n for n in kinfit.PARAMS if n not in ('m1c', 'm1s', 'sigma', 'sigma_los', 'dmu0', 'sigma_mu', 'vslope')
                             + kinfit.TILT + ('qlos',) + kinfit.TURN)
            p = kinfit.fit(d, free=free_all, p0=p, ellipsoid=True)[0]
        # Errors: inverse Hessian of a refit of the rotation curve, k, f and the ellipsoid, started at the best fit p.
        free = [n for n in kinfit.PARAMS if n in ('v0', 'R0', 'k', 'logit_fout') + kinfit.ELLIPSOID]
        _, e, _ = kinfit.fit(d, free=tuple(free), p0=p, ellipsoid=True)
        # g(n): value of parameter n in p (or in the error vector e); the row holds truth, fit and errors [km/s]
        g = lambda n, v=p: v[kinfit.PARAMS.index(n)]  # noqa: E731
        rows.append(dict(mock=name, data=data, **{f'truth_{n}': s for n, s in zip(COMPS, disp)},
                         **{n: g(n) for n in COMPS + ('dlnsig_dR', 'vslope', 'inc', 'theta', 'v0')}, **{f'{n}_err': g(n, e) for n in COMPS}))
        print(rows[-1], flush=True)
```

##### The table and the figure
The fitted dispersions with their inverse-Hessian errors go to `results/kin_ellipsoid_mock.csv`; the figure compares them with
the true values.

```python
# Table of all fits, written to results/kin_ellipsoid_mock.csv
t = pd.DataFrame(rows)
t.to_csv(ROOT / 'results/kin_ellipsoid_mock.csv', index=False, float_format='%.4f')

# Three equal panels (one per dispersion); x: mock index, the three kinds of fit offset by -0.15, 0, +0.15
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
fig, ax = panel_grid(1, 3, width=7.0, left=0.55, right=0.1, hgap=0.5, top=0.45, bottom=0.75)
x = np.arange(len(NAMES))
for a, n, letter in zip(ax[0], COMPS, 'abc'):
    for data, dx, fmt, colour, mfc in (('PM + LOS', -0.15, 'o', 'k', 'k'), ('PM + LOS, tanh', 0.0, 's', '0.55', '0.55'),
                                       ('PM', 0.15, 'o', 'k', 'w')):
        # Fitted dispersion [km/s] with its inverse-Hessian error, mocks in NAMES order
        s = t[t.data == data].set_index('mock').loc[list(NAMES)]
        a.errorbar(x + dx, s[n], yerr=s[f'{n}_err'], fmt=fmt, color=colour, mfc=mfc, ms=4, lw=0.8, label=data)
    # True dispersion of each mock as a red bar
    tr = t[t.data == 'PM + LOS'].set_index('mock').loc[list(NAMES)][f'truth_{n}']
    a.hlines(tr, x - 0.3, x + 0.3, color='C3', lw=1.2, label='truth')
    # Axes: mock names as tick labels, y from 0 to 60 km/s; panel letter in the top left
    a.set(xticks=x, xlim=(-0.5, len(NAMES) - 0.5), ylim=(0, 60), ylabel=LABELS[n] + r' [km s$^{-1}$]')
    a.set_xticklabels(NAMES, rotation=30, ha='right')
    a.text(0.04, 0.92, f'({letter})', transform=a.transAxes, fontsize=7)
ax[0, 0].legend(loc='lower left', bbox_to_anchor=(0, 1.02), ncol=4, fontsize=6, frameon=False)
fig.savefig(ROOT / 'figures/paper_kin_ellipsoid_mock.png')
```

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_kin_ellipsoid_mock.png')

**Figure 6.7.** Recovery of the velocity ellipsoid from physical mocks: fitted $\sigma_R$ (a), $\sigma_\phi$ (b)
and $\sigma_z$ (c) (km/s) for the five mocks (horizontal axis). Filled black circles: proper motions and line-of-sight velocities, the
model of this page; grey squares: the same with a pure tanh rotation curve ($s=0$); open circles: proper motions only. Red bars: true
values. Errors from the inverse Hessian (the curvature of the likelihood at its maximum). Mocks: 'cold thin', $v_c\tanh(R/R_0)$;
'hot', thin with asymmetric drift; 'hot, no drift', thin with $v_c\tanh(R/R_0)$; 'hot thick', with asymmetric drift; 'hot spheroid'
($q$ = 0.5), with asymmetric drift. `scripts/paper_kin_ellipsoid_mock.py`

### Robustness checks of the velocity-ellipsoid model

Chapter 7 tests three assumptions of the model on each giant bin with `scripts/kin_ellipsoid_checks.py`, by maximum likelihood
started from the fit of this page: an inner disc in a plane of its own (its inclination and line of nodes offset by free amounts
inside a fixed radius), a free scale of the internal motion along the line of sight, and one error scale $k$ per group, tested by
fitting the stars brighter and fainter than the bin's median *Gaia* $G$ magnitude separately, each half with its own $k$, rotation
curve and ellipsoid. It writes the log-likelihood gains, the geometry, the rotation speed at $R_\sigma$ and the dispersions of each
fit to `results/kin_ellipsoid_checks.csv`.

<details><summary>About <code>scripts/kin_ellipsoid_checks.py</code></summary>

```text
Robustness checks of the velocity-ellipsoid model for the giant [Fe/H] bins (paper, Sections 4.2 and 4.7; numbers of KB chapter 7
on an inner disc in its own plane, a line-of-sight scale and one Gaia error scale per group, written into results/macros_kinsupport.tex
by scripts/paper_text_kin_support.py).

Maximum-likelihood fits (lmcold.kinfit.fit, ellipsoid=True; proper motions and line-of-sight velocities; release giants in their
measured bins at 2 < rho < 9 deg), each bin on its own:
- 'reference': the Level-2 ellipsoid model as fitted for the paper (fit_kinematics_mcmc.ml, model 'ellipsoid').
- 'tilt': the reference plus an inner disc (rho < kinfit.RHO_TILT) in its own plane (dinc, dtheta free), all else free.
- 'qlos': the reference plus a scale (1 + qlos) of the internal motion along the line of sight, all else free.
- 'bright' and 'faint': the stars brighter and fainter than the bin's median G fitted separately, each with its own Gaia error
  scale k, rotation curve, ellipsoid and outlier fraction (OWN), the geometry, systemic motion and streaming held at the reference:
  a test of one k per group. If k depended on magnitude, the two halves would need different k and, with one k, the dispersions
  would absorb the difference.

Writes results/kin_ellipsoid_checks.csv: population, model, n, lnL (of the stars fitted), dlnL (model minus reference, same stars;
NaN for the halves), inc, theta, dinc, qlos, vrot4 = v0 tanh(4 kpc / R0) [km/s], sigma_R, sigma_phi, sigma_z [km/s], dlnsig_dR,
k, G_split.

Inputs: the giant catalogues of lmcold.samples.giants() (Zenodo bundle 1). No posterior samples: every fit here is a maximum-
likelihood fit started as in scripts/fit_kinematics_mcmc.py.

Run: .venv/bin/python scripts/kin_ellipsoid_checks.py
```
</details>

*Shown, not run here: the giant catalogues of Zenodo bundle 1; maximum-likelihood fits of every giant bin (run time not recorded). The cells below are the script itself; run it from the repository with* `python scripts/kin_ellipsoid_checks.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The parameters left free in the alternative models, and those refitted in each magnitude half.

```python
import numpy as np
import pandas as pd

from fit_kinematics_mcmc import COLS, ml, ring
from lmcold import kinfit, samples
from lmcold.sources import ROOT

P = kinfit.PARAMS.index
# Free parameters of the ellipsoid model in the paper's fits (fit_kinematics_mcmc FIXED['ellipsoid'] held, vlos0 free).
FREE = tuple(n for n in kinfit.BASE + ('vlos0',) + kinfit.STREAM if n != 'sigma') + kinfit.ELLIPSOID + ('vslope',)
OWN = ('v0', 'R0', 'vslope', 'k', 'logit_fout') + kinfit.ELLIPSOID  # refitted in each magnitude half
```

#### One row of the output table
`row` collects the geometry, rotation speed at 4 kpc and dispersions of one fit.

```python
def row(pop, model, d, p, nll, n, **extra):
    """One output row (dict) for parameter vector p (shape (33,), kinfit.PARAMS order) of population pop.

    nll is the negative log likelihood at p, n the number of stars fitted; extra adds columns (dlnL, G_split). d is unused.
    vrot4 [km/s] is v0 tanh(R_SIG / R0), the rotation speed at kinfit.R_SIG = 4 kpc; dispersions in km/s, angles in deg.
    """
    g = lambda name: float(p[P(name)])  # noqa: E731
    return dict(population=pop, model=model, n=n, lnL=-nll, inc=g('inc'), theta=g('theta'), dinc=g('dinc'), qlos=g('qlos'),
                vrot4=g('v0') * np.tanh(kinfit.R_SIG / g('R0')), sigma_R=g('sigma_R'), sigma_phi=g('sigma_phi'),
                sigma_z=g('sigma_z'), dlnsig_dR=g('dlnsig_dR'), k=g('k')) | extra
```

#### Load the giants and fit each bin
The giants of each [Fe/H] bin (Zenodo bundle 1, through `lmcold.samples.giants()`) at 2-9 deg: the reference fit, the two
extended models started from it, and the two magnitude halves. The table goes to `results/kin_ellipsoid_checks.csv`.

```python
rows = []
for label, cat in samples.giants():
    pop = f'giants {label}'
    c = ring(cat | dict(dmu=np.full(len(cat['ra']), np.nan)))
    d = kinfit.data(*(c[k] for k in COLS))
    p0 = ml(d, 'ellipsoid')   # reference: the maximum-likelihood start of the paper's MCMC fit
    nll0 = float(kinfit.neg_loglike(p0, d, True))
    rows.append(row(pop, 'reference', d, p0, nll0, len(c['ra']), dlnL=0.0))
    for model, extra in (('tilt', kinfit.TILT), ('qlos', ('qlos',))):
        p, _, nll = kinfit.fit(d, free=FREE + extra, p0=p0, ellipsoid=True)
        rows.append(row(pop, model, d, p, nll, len(c['ra']), dlnL=nll0 - nll))
        print(rows[-1], flush=True)
    gsplit = float(np.median(c['G']))   # Gaia G [mag]: median of the bin, the split between bright and faint halves
    for half, keep in (('bright', c['G'] < gsplit), ('faint', c['G'] >= gsplit)):
        h = {k: np.asarray(v)[keep] for k, v in c.items()}
        dh = kinfit.data(*(h[k] for k in COLS))
        p, _, nll = kinfit.fit(dh, free=OWN, p0=p0, ellipsoid=True)
        rows.append(row(pop, half, dh, p, nll, int(keep.sum()), dlnL=np.nan, G_split=gsplit))
        print(rows[-1], flush=True)
pd.DataFrame(rows).to_csv(ROOT / 'results/kin_ellipsoid_checks.csv', index=False, float_format='%.4f')
```

### The sky-aligned dispersion model

Our first fits used the mean motion of the model section with a pure tanh rotation curve, $v_{\rm rot}=v_0\tanh(R/R_0)$, and two
dispersions per group, constant over the disc: $\sigma$ (written $\sigma_{\rm PM}$ where it must be told apart), the same in both
proper-motion directions, and $\sigma_{\rm los}$ along the line of sight. In the star's (east, north, line-of-sight) frame the random
velocities then have covariance ${\rm diag}(\sigma^2,\sigma^2,\sigma_{\rm los}^2)$, an ellipsoid aligned with the line of sight, not
with the disc, and the likelihood factorises into a proper-motion and a line-of-sight term:

$$
\begin{aligned}
\mathcal{L}_i={}&(1-f)\,\mathcal{N}\bigl(\boldsymbol\mu_i\,|\,\boldsymbol\mu_{\rm m},\,k^2C_i+\sigma_\mu^2I\bigr)\,
\mathcal{N}\bigl(v_i\,|\,v_{\rm m},\,e_i^2+\sigma_{\rm los}^2\bigr)\\
&+f\,\mathcal{N}\bigl(\boldsymbol\mu_i\,|\,\boldsymbol\mu_{\rm m},\,k^2C_i+\sigma_\mu^2I+S^2I\bigr)\,
\mathcal{N}\bigl(v_i\,|\,v_{\rm m},\,e_i^2+\sigma_{\rm los}^2+\kappa^2D_i^2S^2\bigr),
\end{aligned}
$$

with $\boldsymbol\mu_i$ and $v_i$ the measured proper motion and line-of-sight velocity of star $i$ and $\sigma_\mu=\sigma/(\kappa D_i)$
the dispersion in proper motion at the star's distance. Neither dispersion is a component of the disc-frame ellipsoid, but each is
close to a combination of them: averaged over azimuth, $\sigma_{\rm los}^2\simeq\sigma_z^2\cos^2i+\tfrac12\sin^2i\,(\sigma_R^2+\sigma_\phi^2)$
and $\sigma^2\simeq\tfrac12(1-\tfrac12\sin^2i)(\sigma_R^2+\sigma_\phi^2)+\tfrac12\sigma_z^2\sin^2i$ (from $\sigma_{\rm los}^2(\phi)$ above
and the trace of the ellipsoid, with the central line of sight for every star). The hot thin mock with asymmetric drift, with
$(\sigma_R,\sigma_\phi,\sigma_z)$ = (40, 28, 30) km/s, gives $\sigma$ =
34.5 and $\sigma_{\rm los}$ = 31.6 km/s against 34.1 and 31.1
km/s from these averages at its inclination of 28°. Because a real ellipsoid projects differently at each
azimuth, these dispersions rank the groups but do not measure $\sigma_R$ or $\sigma_z$; the velocity-ellipsoid model fits better in
every group, by $\Delta\ln L$ = 37, 110, 440, 1 443 and
465 for the giant bins and 73 for the RR Lyrae at 2° < ρ < 9°,
and 25, 73, 289, 663, 218 and
26 at 4° < ρ < 9° (log-likelihood of each group's stars at the best of
300 posterior draws of each model; `results/kin_model_comparison.csv`, `scripts/kin_model_comparison.py`).

The joint Level-2 fit with sky-aligned dispersions puts the centre 0.530° east and 0.024° south of the
vdMK14 centre, with $i$ = 28.0°, $\theta$ = 142.7° and $m_{2c}$ = -0.116,
0.06° in $\Delta x$ and 1.2° in $\theta$ from the values of Table 6.2. Table 6.3
gives the rotation and dispersions of each group fitted on its own (stage 1). The maps of $v_t$, $v_r$ and $v_{\rm los}$ in the frame
of the joint fit (Figure 6.8, Figure 6.9 and Figure 6.10), the data against position angle
(Figure 6.11) and the residuals of the three models against position angle (Figure 6.12)
show for this model what Figure 6.1–Figure 6.4 and
Figure 6.5 show for the velocity ellipsoid. The rotational support $v_0/\sigma_{\rm los}$ rises from 1.84
to 4.64 from the metal-poor to the metal-rich giants (APOGEE alone: 5.99), the same ordering as
with the velocity ellipsoid (chapter 7). With a pure tanh curve the radial profile of the residuals shows a decline beyond 7°: the
three metal-rich bins and the RR Lyrae rotate 4–11 km/s slower than the model there (the
two metal-poor bins by 0.7–2.5 km/s; `results/kin_residual_radial.csv`, not shown), which
the outer slope $s$ of the model section removes. The sky-aligned fits sampled the posteriors with $\hat R\leq$ 1.004
and at least 1 034 effective draws per parameter; the RR Lyrae fit at 4° < ρ <
9° had 7 divergent transitions.

In [ ]:
import numpy as np
import pandas as pd
# Rotational support per population (S) and the stage-1 posterior of the streaming fit (F)
S = pd.read_csv('results/kin_support.csv').set_index('population')
F = pd.read_csv('results/kin_mcmc_data_stream.csv').query('stage == 1')


def q(v, lo, hi, d):
    """Median with its 16th-84th percentile range."""
    return f'{v:.{d}f} ({lo:.{d}f}–{hi:.{d}f})'


def stage1(pop, p):
    r = F[(F.population == pop) & (F.param == p)].iloc[0]
    return r['median'], r.p16, r.p84


# One row per population: rotation amplitude v0, turnover radius R0 of v0 tanh(R/R0), dispersions, their ratios, outlier fraction f = 1/(1 + exp(-logit_fout))
rows = []
for pop, s in S.iterrows():
    rr = pop == 'RR Lyrae'
    rows.append((pop.replace('giants ', ''), f'{int(s.n):,}', q(s.v0, s.v0_p16, s.v0_p84, 1), q(*stage1(pop, 'R0'), 2),
                 '–' if rr else q(s.sigma_los, s.sigma_los_p16, s.sigma_los_p84, 1),
                 '–' if rr else q(s.ratio, s.ratio_lo, s.ratio_hi, 2), q(s.sigma, s.sigma_p16, s.sigma_p84, 1),
                 q(s.ratio_pm, s.ratio_pm_lo, s.ratio_pm_hi, 2), f'{100 / (1 + np.exp(-stage1(pop, "logit_fout")[0])):.1f}'))
# Rows from the APOGEE line-of-sight velocities alone, where measured
for pop, s in S[np.isfinite(S.v0_apogee)].iterrows():
    rows.append(('APOGEE ' + pop.replace('giants ', ''), f'{int(s.n_apogee):,}', f'{s.v0_apogee:.1f} ± {s.v0_apogee_err:.1f}', '',
                 f'{s.sigma_los_apogee:.1f} ± {s.sigma_los_apogee_err:.1f}', f'{s.ratio_apogee:.2f} ± {s.ratio_apogee_err:.2f}', '', '', ''))
pd.DataFrame(rows, columns=['Group', 'N', 'v0 (km/s)', 'R0 (kpc)', 'σ_los (km/s)', 'v0/σ_los', 'σ_PM (km/s)', 'v0/σ_PM',
                            'f (%)']).set_index('Group')

**Table 6.3.** Level-2 parameters of each group with sky-aligned dispersions, fitted on its own (stage 1) at
2° < ρ < 9°: posterior median with the 16th–84th percentile range in brackets
($v_0/\sigma_{\rm los}$ from the posterior draws). $N$: stars; $f$: outlier fraction. $\sigma_{\rm PM}$ and $v_0/\sigma_{\rm PM}$: the
same fits at 4° < ρ < 9° (1 793, 2 798, 7 540,
19 793 and 7 226 giants, 7 200 RR Lyrae), where one error scale $k$ describes the Gaia
errors; the ratio from the draws of those fits, whose $v_0$ is 56.7, 58.1,
62.3, 71.6 and 76.1 km/s for the giants and 63.9 km/s for
the RR Lyrae. Last three rows: APOGEE giants alone, maximum-likelihood fits of $v_0$, $\sigma_{\rm los}$, $k$ and $f$ with the other
parameters held at the posterior medians of the full bin (errors from the inverse Hessian, conditional on the fixed geometry,
streaming and $R_0$). From `results/kin_support.csv` (`scripts/paper_kin_support.py`) and `results/kin_mcmc_data_stream.csv`.

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_kin_maps_vt.png')

**Figure 6.8.** As Figure 6.1 for the model with sky-aligned dispersions: frame and model of its
joint Level-2 fit at 2° < ρ < 9° for all groups, RR Lyrae included, and residual significance
with the scatter of the sky-aligned likelihood. Fitted centre (α, δ) = (81.40°, -69.61°).
`scripts/paper_kin_residuals.py` (without argument)

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_kin_maps_vr.png')

**Figure 6.9.** As Figure 6.8 for the radial component $v_r$ (km/s). `scripts/paper_kin_residuals.py`

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_kin_maps_vlos.png')

**Figure 6.10.** As Figure 6.8 for the line-of-sight velocity $v_{\rm los}$ (km/s), giants only.
`scripts/paper_kin_residuals.py`

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_kin_data_profiles.png')

**Figure 6.11.** As Figure 6.4 in the frame of the joint Level-2 fit with
sky-aligned dispersions. `scripts/paper_kin_residuals.py`

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_kin_residual_profiles.png')

**Figure 6.12.** As Figure 6.5, with sky-aligned dispersions in all three
columns (the joint Level-2 fit in the right column, g–i). `scripts/paper_kin_residuals.py`

#### Comparing the two models

The log-likelihood gains $\Delta\ln L$ of the velocity-ellipsoid model over the sky-aligned model quoted above are computed by
`scripts/kin_model_comparison.py`. For each group and each radial range it evaluates the log likelihood of the group's stars at the
same random subset of posterior draws of its stage-1 fit under each model, read from the posterior samples, and takes the difference
of the best (and of the median) values; no fit is repeated. It writes `results/kin_model_comparison.csv`.

<details><summary>About <code>scripts/kin_model_comparison.py</code></summary>

```text
Log-likelihood gain of the velocity-ellipsoid model over the sky-aligned dispersion model (paper, appendix app:kin_sky; the
Delta ln L values of KB chapter 6, 'Tests' and 'The sky-aligned dispersion model', written into results/macros_kinmodel.tex by
scripts/paper_text_kin_model.py).

For each population and radial range (2 < rho < 9 deg and OUTER_INNER < rho < 9 deg) the log likelihood of the population's own
stars (lmcold.kinfit.neg_loglike; ellipsoid=True for the ellipsoid model) is evaluated at NDRAW posterior draws of its stage-1 fit
(each population on its own) of each model, read from data/kin_mcmc_data_<run>_samples.npz (scripts/fit_kinematics_mcmc.py:
runs 'stream' and 'ellipsoid', and their '_outer' versions). Parameters not sampled stay at their kinfit.start() values, as in the
fits. No refitting: the best of the draws stands in for the maximum likelihood (it agrees with a maximum-likelihood refit to 1 in
ln L where both were computed).

Writes results/kin_model_comparison.csv: rho_inner [deg], population, n (stars), dlnL_max (best ellipsoid draw minus best sky-aligned
draw), dlnL_median (same with the median over draws), dk (extra parameters of the ellipsoid model: sigma_R, sigma_phi, sigma_z,
dlnsig_dR, vslope replace sigma, sigma_los; 3 for the giants, 4 for the RR Lyrae, which have no sigma_los).

Inputs: the posterior samples data/kin_mcmc_data_{stream,ellipsoid}[_outer]_samples.npz (Zenodo bundle 1) and the star catalogues
of lmcold.samples (giants and RR Lyrae, also bundle 1).

Run: .venv/bin/python scripts/kin_model_comparison.py   (after fit_kinematics_mcmc.py data stream|ellipsoid [outer])
```
</details>

*Shown, not run here: the posterior samples of the stream and ellipsoid fits and the star catalogues, all in Zenodo bundle 1 (run time not recorded). The cells below are the script itself; run it from the repository with* `python scripts/kin_model_comparison.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

##### Settings
The number of posterior draws at which the likelihood is evaluated, and their indices (the same for every fit).

```python
import jax
import numpy as np
import pandas as pd

from fit_kinematics_mcmc import COLS, OUTER_INNER, ring
from lmcold import kinfit, samples
from lmcold.sources import DATA, ROOT

NDRAW = 300  # posterior draws per model and population
DRAWS = np.random.default_rng(0).choice(4000, NDRAW, replace=False)  # same draw indices for every fit (4 chains x 1000)
```

##### Log-likelihood at the posterior draws
`lnl` reads the stage-1 draws of one population from a run's samples file and evaluates the log likelihood of its stars at each.

```python
def lnl(run, i, d, ellipsoid):
    """ln L of data d at the NDRAW stage-1 draws of population i in run.

    Parameters
    ----------
    run : str
        Run name of fit_kinematics_mcmc.py without 'data_' ('stream', 'ellipsoid', 'stream_outer', 'ellipsoid_outer').
    i : int
        Population index in the run (giant bins 0-4 metal-poor first, RR Lyrae 5).
    d : dict
        kinfit.data() arrays of that population's stars.
    ellipsoid : bool
        Use the velocity-ellipsoid likelihood (True for the 'ellipsoid' runs).

    Returns
    -------
    ndarray, shape (NDRAW,)
        ln L of all stars of d at each draw.
    """
    z = np.load(DATA / f'kin_mcmc_data_{run}_samples.npz')   # keys 'stage|population|site', arrays of posterior draws
    # Parameter vectors, shape (NDRAW, len(kinfit.PARAMS)), starting at the kinfit.start() values (kept for unsampled parameters).
    p = np.tile(kinfit.start(), (NDRAW, 1))
    # Overwrite each sampled parameter of stage 1, population i, with its draws at the indices DRAWS.
    for key in z.files:
        if key.startswith(f'1|{i}|'):
            n = key.split('|')[2]   # site name; a population's own parameters carry the suffix _0
            p[:, kinfit.PARAMS.index(n[:-2] if n.endswith('_0') else n)] = z[key].reshape(-1)[DRAWS]
    # Compiled ln L of the stars d as a function of the parameter vector; evaluated at each draw.
    f = jax.jit(lambda q: -kinfit.neg_loglike(q, d, ellipsoid))
    return np.array([float(f(q)) for q in p])
```

##### Load the stars and compare the two models
For each radial range (2-9 deg, and 4-9 deg for the '_outer' runs) the giants of the five [Fe/H] bins and the RR Lyrae are read
through `lmcold.samples` (Zenodo bundle 1) and cut by `ring`; for each population the best and the median ln L over the draws
of the two models are compared and written to `results/kin_model_comparison.csv`.

```python
rows = []
# Inner radius [deg] of the ring and run-name suffix: 2-9 deg ('' runs) and OUTER_INNER-9 deg ('_outer' runs).
for inner, suffix in ((2.0, ''), (OUTER_INNER, '_outer')):
    # Giant bins (no distances: dmu NaN), then the RR Lyrae (no line-of-sight velocities: vlos, evlos NaN), cut to the ring.
    pops = [(f'giants {lab}', ring(cat | dict(dmu=np.full(len(cat['ra']), np.nan)), inner)) for lab, cat in samples.giants()]
    rr = samples.rrab()
    nan = np.full(len(rr['ra']), np.nan)
    pops.append(('RR Lyrae', ring(rr | dict(vlos=nan, evlos=nan, dmu=nan), inner)))
    for i, (name, c) in enumerate(pops):
        d = kinfit.data(*(c[k] for k in COLS))   # likelihood inputs of this population
        # ln L at the draws, shape (NDRAW,): sky-aligned dispersion model ('stream' runs) and ellipsoid model.
        sky, ell = lnl('stream' + suffix, i, d, False), lnl('ellipsoid' + suffix, i, d, True)
        # Gains of the ellipsoid model: best draw minus best draw, median minus median; dk extra parameters (see docstring).
        rows.append(dict(rho_inner=inner, population=name, n=len(c['ra']), dlnL_max=ell.max() - sky.max(),
                         dlnL_median=np.median(ell) - np.median(sky), dk=4 if name == 'RR Lyrae' else 3))
        print(rows[-1], flush=True)
pd.DataFrame(rows).to_csv(ROOT / 'results/kin_model_comparison.csv', index=False, float_format='%.1f')
```

### Correlations of the geometric parameters

Figure 6.13 shows the posteriors of the centre, the systemic proper motion north, the orientation and the $m=2$ part of
$v_\phi$ in the sky-aligned Level-2 fits of each group on its own and of all groups together. The centre offset $\Delta x$ and
$\mu_{\delta,0}$ are correlated in every group along one line: a centre further east goes with a larger northward systemic proper
motion. The script also writes a full corner plot of every stage-1 fit (`figures/fig_kin_mcmc_corner_data_stream_*.png`).

In [ ]:
import sys
sys.argv = ['paper_kin_corner.py', 'data_stream']   # as: python scripts/paper_kin_corner.py data_stream

<details><summary>About <code>scripts/paper_kin_corner.py</code></summary>

```text
Corner plots of a kinematic posterior: argument RUN, samples data/kin_mcmc_<RUN>_samples.npz from scripts/fit_kinematics_mcmc.py.

figures/fig_kin_mcmc_corner_<RUN>_shared.png: the sampled parameters the joint fit shares (of centre offset, bulk proper motion,
line-of-sight systemic velocity, inclination, line of nodes, lopsided part), stage 1 for each population in its own colour and
stage 2 (all populations together) in black; 1 and 2 sigma contours. For RUN data_stream also figures/paper_kin_corner.png
(paper appendix): the same for the parameters PAPER only (centre, proper motion north, orientation, m = 2 part of v_phi). figures/fig_kin_mcmc_corner_<RUN>_<i>.png: all sampled
parameters of stage 1 for population i (outlier fraction shown as f_out, not its logit).

RUN is '<kind>_<model>' of fit_kinematics_mcmc.py, e.g. data_centre, data_lopsided, data_both (suffix _dist for the runs with RR
Lyrae distance moduli). The npz keys are
'stage|population|site': '1|<i>|<site>' for the stage-1 fit of population i (its own parameters have the site suffix _0,
since each stage-1 fit holds one population) and '2|all|<site>' for the joint fit, each of shape (chains, draws) (flat in runs
before 2026-10-29; reshape(-1) reads both); 'names' lists the populations in order. The per-population and full shared plots are online
material. Contours: 2D levels enclosing 39% and 86% of the mass (1 and 2 sigma of a 2D Gaussian); bins 40,
smoothing 1 bin. In the shared plot each 1D histogram is normalised to unit area, and the axis ranges are the 0.2-99.8 percentiles
of all fits together.

Used in KB chapter 6 ('Correlations of the geometric parameters'), run with data_stream. Its input for that run,
data/kin_mcmc_data_stream_samples.npz, is in Zenodo bundle 1, so the figure is made without rerunning the MCMC fit.

Run: .venv/bin/python scripts/paper_kin_corner.py data_stream
```
</details>

**Data read by `scripts/paper_kin_corner.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/kin_mcmc_data_stream_samples.npz` | Zenodo bundle 1 | Posterior samples of the sky-aligned kinematic fit, 2-9 deg (NUTS) |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import sys

import matplotlib
matplotlib.use('Agg')
import corner
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

from lmcold import kinfit
from lmcold.sources import DATA, ROOT

LABELS = dict(dx=r'$\Delta x$ E [deg]', dy=r'$\Delta y$ N [deg]', pmra0=r'$\mu_{\alpha*,0}$ [mas/yr]',
              pmdec0=r'$\mu_{\delta,0}$ [mas/yr]', inc=r'$i$ [deg]', theta=r'$\theta$ [deg]', v0=r'$v_0$ [km/s]',
              R0=r'$R_0$ [kpc]', sigma=r'$\sigma$ [km/s]', k=r'$k$', logit_fout=r'$f_{\rm out}$',
              vlos0=r'$v_{\rm los,0}$ [km/s]', sigma_los=r'$\sigma_{\rm los}$ [km/s]', m1c=r'$m_{1c}$', m1s=r'$m_{1s}$',
              dmu0=r'$\Delta\mu_0$ [mag]', sigma_mu=r'$\sigma_\mu$ [mag]', m2c=r'$m_{2c}$', m2s=r'$m_{2s}$',
              vr0=r'$v_{R,0}$', r2c=r'$r_{2c}$', r2s=r'$r_{2s}$')
LEVELS = 1 - np.exp(-.5 * np.array([1., 4.]))  # 1 and 2 sigma in 2D: mass fractions 0.39 and 0.86 (1 - exp(-n^2 / 2))
GIANT_COLOURS = ['#d7191c', '#fdae61', '#a6d96a', '#1a9641', '#2878b5']  # [Fe/H] bins, metal-poor first; RR Lyrae purple
RUN = sys.argv[1]  # '<kind>_<model>', e.g. data_both
PAPER = ('dx', 'dy', 'pmdec0', 'inc', 'theta', 'm2c', 'm2s')  # parameters of the paper's reduced shared corner plot

#### Load the data
The posterior samples of the run given as argument (`data_stream` for the paper): `data/kin_mcmc_<RUN>_samples.npz` (from
`scripts/fit_kinematics_mcmc.py`) holds the draws of every stage-1 fit and of the joint stage-2 fit, and the population names.

In [ ]:
# Posterior draws of run RUN: keys 'stage|population|site', each of shape (chains, draws), plus 'names'.
s = np.load(DATA / f'kin_mcmc_{RUN}_samples.npz')
names = list(s['names'])  # the populations, in the order of the run

#### Colours and shared parameters
Each population gets a fixed colour, and the shared parameters sampled in every fit are listed.

In [ ]:
# One colour per population, fixed by name so that it does not depend on the population order in the run.
colours = [GIANT_COLOURS[[m for m in names if m != 'RR Lyrae'].index(n)] if n != 'RR Lyrae' else '#7b3294' for n in names]
# (draws, len(params)) array of the samples under key.format(name) for each parameter name.
stack = lambda key, params: np.column_stack([s[key.format(n)].reshape(-1) for n in params])  # noqa: E731
# Shared parameters drawn in every fit of this run (e.g. vlos0 is held in the stage-1 RR Lyrae fit, which has no velocities).
SHARED = [n for n in kinfit.SHARED if all(f'{k}|{n}' in s for k in ['2|all'] + [f'1|{i}' for i in range(len(names))])]  # sampled in every fit


def overlay(params, fname, dpi):
    """Corner plot of params: each stage-1 fit in its colour, then the joint fit in black, on common axis ranges."""
    shared = [stack(f'1|{i}|{{}}', params) for i in range(len(names))] + [stack('2|all|{}', params)]
    lo, hi = np.percentile(np.vstack(shared), [.2, 99.8], axis=0)
    fig = None
    for x, colour in zip(shared, colours + ['k']):
        fig = corner.corner(x, fig=fig, range=list(zip(lo, hi)), bins=40, smooth=1., levels=LEVELS, color=colour,
                            plot_datapoints=False, plot_density=False, fill_contours=False, hist_kwargs=dict(density=True),
                            labels=[LABELS[n] for n in params], label_kwargs=dict(fontsize=11))
    for j in range(len(params)):  # 1D histograms of different widths: rescale each diagonal panel to its tallest curve
        ax = fig.axes[j * (len(params) + 1)]
        ax.relim(), ax.autoscale_view(scalex=False)
    fig.legend([Line2D([], [], color=c) for c in colours + ['k']], names + ['joint (stage 2)'], loc='upper right',
               bbox_to_anchor=(.98, .98), frameon=False, fontsize=12)
    fig.savefig(ROOT / f'figures/{fname}', dpi=dpi)
    plt.close(fig)

#### The shared parameters, all fits overlaid
All shared parameters (online material) and, for the sky-aligned run, the paper's reduced set `PAPER`.

In [ ]:
overlay(SHARED, f'fig_kin_mcmc_corner_{RUN}_shared.png', 110)
if RUN == 'data_stream':  # the paper's appendix figure: the geometry and m = 2 terms only
    overlay([n for n in PAPER if n in SHARED], 'paper_kin_corner.png', 110)

#### Every parameter of each stage-1 fit
One full corner plot per population (online material).

In [ ]:
# One full corner plot per population (stage 1), with medians and 16th/84th percentiles in the panel titles.
for i, name in enumerate(names):
    key = lambda n: f'1|{i}|{n}' if n in kinfit.SHARED else f'1|{i}|{n}_0'  # noqa: E731  (own parameters: site suffix _0)
    params = [n for n in kinfit.PARAMS if key(n) in s]
    x = np.column_stack([s[key(n)].reshape(-1) for n in params])
    j = params.index('logit_fout')
    x[:, j] = 1 / (1 + np.exp(-x[:, j]))  # logit -> outlier fraction f_out
    fig = corner.corner(x, bins=40, smooth=1., levels=LEVELS, color=colours[i], plot_datapoints=False, quantiles=[.16, .5, .84],
                        show_titles=True, title_fmt='.3g', labels=[LABELS[n] for n in params], label_kwargs=dict(fontsize=11))
    fig.suptitle(f'{name}, stage 1', fontsize=14, x=.7, y=.9)
    fig.savefig(ROOT / f'figures/fig_kin_mcmc_corner_{RUN}_{i}.png', dpi=90)
    plt.close(fig)

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_kin_corner.png')

**Figure 6.13.** Posteriors of the centre offset east and north ($\Delta x$, $\Delta y$, deg), the systemic proper
motion north ($\mu_{\delta,0}$, mas/yr), the inclination $i$ and line of nodes $\theta$ (deg) and the $m=2$ part of $v_\phi$ ($m_{2c}$,
$m_{2s}$) in the Level-2 fits with sky-aligned dispersions of each group on its own (colours, stage 1, legend top right; RR Lyrae
at 2° < ρ < 9°) and of all groups together (black, stage 2). Diagonal: one-dimensional
posteriors, each normalised to unit area; off-diagonal: contours enclosing 39 and 86 per cent of
each two-dimensional posterior. `scripts/paper_kin_corner.py data_stream`